# Đề tài 13 – Đánh giá rủi ro tín dụng BNPL (bản gộp chạy trên Kaggle)
Chạy lần lượt từ trên xuống (Run All). Cần Add Input: cuộc thi *home-credit-credit-risk-model-stability*.
Khi nộp bài: tắt Internet trong Settings của notebook. Phần tạo `submission.csv` đặt cuối cùng; mỗi phần giải phóng bộ nhớ của phần trước (v6.2).

In [ ]:
%%writefile utils.py
"""Ham dung chung cho De tai 13 - Credit Scoring BNPL.

Du lieu: Home Credit - Credit Risk Model Stability (Kaggle 2024).
"""
import glob
import os
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import (average_precision_score, brier_score_loss,
                             roc_auc_score, roc_curve)

KAGGLE_DIR = "/kaggle/input/home-credit-credit-risk-model-stability/parquet_files/train"
KEYS = ("case_id", "WEEK_NUM", "MONTH", "target", "num_group1", "num_group2")
META = ["case_id", "date_decision", "MONTH", "WEEK_NUM", "target"]
SPLIT = "train"      # "test" khi du doan cho tap test cua cuoc thi (notebook 07)


# ---------------------------------------------------------------- duong dan
def get_data_dir() -> Path:
    """Thu muc chua train_*.parquet: bien moi truong HC_DATA_DIR > Kaggle > Data/raw."""
    if os.environ.get("HC_DATA_DIR"):
        return Path(os.environ["HC_DATA_DIR"])
    if Path(KAGGLE_DIR).exists():
        return Path(KAGGLE_DIR)
    if Path("/kaggle/input").exists():          # tu do thu muc chua train_base.parquet
        hits = glob.glob("/kaggle/input/**/train_base.parquet", recursive=True)
        if hits:
            return Path(hits[0]).parent
        raise FileNotFoundError("Khong thay train_base.parquet trong /kaggle/input. "
                                "Kiem tra Notebook da Add Input cuoc thi home-credit-credit-risk-model-stability chua.")
    return Path(__file__).resolve().parent.parent / "Data" / "raw"


def get_out_dir() -> Path:
    """Thu muc luu file trung gian: HC_OUT_DIR > /kaggle/working > Data/processed."""
    if os.environ.get("HC_OUT_DIR"):
        out = Path(os.environ["HC_OUT_DIR"])
    elif Path("/kaggle/working").exists():
        out = Path("/kaggle/working")
    else:
        out = Path(__file__).resolve().parent.parent / "Data" / "processed"
    out.mkdir(parents=True, exist_ok=True)
    return out


def table_files(name: str) -> list:
    """Cac file parquet cua mot bang, vd 'static_0' -> train_static_0_0.parquet, ...
    SPLIT = "test" doc test_*.parquet trong thu muc .../parquet_files/test."""
    d = get_data_dir()
    if SPLIT != "train":
        d = d.parent / SPLIT
    files = sorted(glob.glob(str(d / f"{SPLIT}_{name}.parquet")) +
                   glob.glob(str(d / f"{SPLIT}_{name}_[0-9]*.parquet")))
    if not files:
        raise FileNotFoundError(f"Khong tim thay bang '{name}' trong {d}. Xem Data/README.txt")
    return files


# ---------------------------------------------------------------- doc du lieu
def _to_date(c, dt):
    if dt == pl.String:
        return pl.col(c).str.to_date(strict=False)
    return pl.col(c).cast(pl.Date)


def cast_by_suffix(lf: pl.LazyFrame) -> pl.LazyFrame:
    """Ep kieu theo hau to ten cot cua bo du lieu: P/A = so, M = chuoi, D = ngay."""
    exprs = []
    for c, dt in lf.collect_schema().items():
        if c in KEYS:
            exprs.append(pl.col(c).cast(pl.Int64))
        elif c == "date_decision" or c[-1] == "D":
            exprs.append(_to_date(c, dt))
        elif c[-1] in "PA":
            exprs.append(pl.col(c).cast(pl.Float64))
        elif c[-1] == "M":
            exprs.append(pl.col(c).cast(pl.String))
    return lf.with_columns(exprs)


def scan_table(name: str) -> pl.LazyFrame:
    parts = [cast_by_suffix(pl.scan_parquet(f)) for f in table_files(name)]
    return parts[0] if len(parts) == 1 else pl.concat(parts, how="diagonal_relaxed")


def load_base(sample_frac=None, seed=42) -> pl.LazyFrame:
    """Bang goc. sample_frac: lay mau ho so theo hash case_id (du lieu that, khong sinh them)."""
    base = scan_table("base")
    if sample_frac is not None and sample_frac < 1:
        base = base.filter(pl.col("case_id").hash(seed) % 1000 < int(sample_frac * 1000))
    return base


def collect(lf: pl.LazyFrame) -> pl.DataFrame:
    try:
        return lf.collect(engine="streaming")
    except TypeError:
        return lf.collect()


# ---------------------------------------------------------------- dac trung
def _dates_to_days(lf: pl.LazyFrame) -> pl.LazyFrame:
    """Doi cot ngay thanh so ngay so voi date_decision (am = truoc ngay quyet dinh)."""
    cols = [c for c, dt in lf.collect_schema().items() if dt == pl.Date and c != "date_decision"]
    return lf.with_columns(
        [(pl.col(c) - pl.col("date_decision")).dt.total_days().cast(pl.Float32) for c in cols]
    ).drop("date_decision")


def _with_decision_date(name: str, base: pl.LazyFrame) -> pl.LazyFrame:
    keys = base.select("case_id", "date_decision")
    return _dates_to_days(scan_table(name).join(keys, on="case_id", how="inner"))


def depth0_features(name: str, base: pl.LazyFrame) -> pl.LazyFrame:
    """Bang 1 dong / ho so."""
    return _with_decision_date(name, base)


def depth1_features(name: str, base: pl.LazyFrame) -> pl.LazyFrame:
    """Bang nhieu dong / ho so -> gop ve 1 dong: so ban ghi, max, mean, gia tri dau."""
    lf = _with_decision_date(name, base)
    schema = lf.collect_schema()
    sort_col = "num_group1" if "num_group1" in schema else None
    exprs = [pl.len().alias(f"{name}__count")]
    for c, dt in schema.items():
        if c in KEYS:
            continue
        if dt == pl.Boolean:
            exprs.append(pl.col(c).cast(pl.Float32).mean().alias(f"{name}__{c}__mean"))
        elif dt.is_numeric():
            exprs.append(pl.col(c).max().alias(f"{name}__{c}__max"))
            exprs.append(pl.col(c).mean().alias(f"{name}__{c}__mean"))
        elif dt == pl.String:
            exprs.append(pl.col(c).first().alias(f"{name}__{c}__first"))
    if sort_col:
        lf = lf.sort("case_id", sort_col)
    return lf.group_by("case_id").agg(exprs)


def build_features(depth0=(), depth1=(), sample_frac=None) -> pl.DataFrame:
    base = load_base(sample_frac)
    out = base
    for name in depth0:
        out = out.join(depth0_features(name, base), on="case_id", how="left")
    for name in depth1:
        out = out.join(depth1_features(name, base), on="case_id", how="left")
    return collect(out)


def prune_features(df: pl.DataFrame, max_null=0.95, max_card=200) -> pl.DataFrame:
    """Bo cot gan nhu rong, cot hang so, cot chuoi qua nhieu gia tri; ep kieu gon bo nho."""
    n = df.height
    drop = []
    for c in df.columns:
        if c in META:
            continue
        s = df[c]
        if s.null_count() / n > max_null:
            drop.append(c)
        elif s.dtype == pl.String:
            k = s.n_unique()
            if k <= 1 or k > max_card:
                drop.append(c)
        elif s.dtype.is_numeric() and s.n_unique() <= 1:
            drop.append(c)
    df = df.drop(drop)
    feats = [c for c in df.columns if c not in META]
    return df.with_columns(
        [pl.col(c).cast(pl.Float32) for c in feats if df[c].dtype.is_numeric() or df[c].dtype == pl.Boolean]
        + [pl.col(c).cast(pl.Categorical) for c in feats if df[c].dtype == pl.String]
    )


# ---------------------------------------------------------------- chia du lieu
def time_split(df: pd.DataFrame, train=0.70, valid=0.15):
    """Chia train/valid/test theo WEEK_NUM (test = cac tuan cuoi)."""
    weeks = np.sort(df["WEEK_NUM"].unique())
    n = len(weeks)
    w_tr = weeks[: int(n * train)]
    w_va = weeks[int(n * train): int(n * (train + valid))]
    w_te = weeks[int(n * (train + valid)):]
    return (df[df["WEEK_NUM"].isin(w_tr)].copy(),
            df[df["WEEK_NUM"].isin(w_va)].copy(),
            df[df["WEEK_NUM"].isin(w_te)].copy())


# ---------------------------------------------------------------- danh gia
def ks_stat(y, p) -> float:
    fpr, tpr, _ = roc_curve(y, p)
    return float(np.max(tpr - fpr))


def evaluate(y, p) -> dict:
    auc = roc_auc_score(y, p)
    return {"AUC": auc, "Gini": 2 * auc - 1, "PR_AUC": average_precision_score(y, p),
            "KS": ks_stat(y, p), "Brier": brier_score_loss(y, np.clip(p, 0, 1))}


def psi(expected, actual, bins=10) -> float:
    """Population Stability Index. >0.1: can theo doi, >0.25: troi manh."""
    expected, actual = np.asarray(expected, float), np.asarray(actual, float)
    cuts = np.unique(np.quantile(expected, np.linspace(0, 1, bins + 1)))
    if len(cuts) < 3:
        return 0.0
    e = np.histogram(expected, cuts)[0] / len(expected)
    a = np.histogram(np.clip(actual, cuts[0], cuts[-1]), cuts)[0] / len(actual)
    e, a = np.clip(e, 1e-6, None), np.clip(a, 1e-6, None)
    return float(np.sum((a - e) * np.log(a / e)))


def gini_by_week(df: pd.DataFrame, score_col="score") -> pd.Series:
    def _g(g):
        if g["target"].nunique() < 2:
            return np.nan
        return 2 * roc_auc_score(g["target"], g[score_col]) - 1
    return df.groupby("WEEK_NUM")[["target", score_col]].apply(_g)


# ---------------------------------------------------------------- WoE / IV
class WoEEncoder(BaseEstimator, TransformerMixin):
    """Ma hoa Weight of Evidence. Cot so: chia bin theo phan vi; gia tri thieu la 1 bin rieng.

    WoE = ln( %good / %bad ) voi good = khong vo no (target=0). Chi fit tren tap train.
    """

    def __init__(self, bins=10):
        self.bins = bins

    def _bin(self, s: pd.Series, col):
        if col in self.edges_:
            b = pd.cut(pd.to_numeric(s, errors="coerce"), self.edges_[col], labels=False)
            return pd.Series(b, index=s.index).fillna(-1).astype(int)
        o = s.astype(object)
        return o.where(o.notna(), "NA").astype(str)

    def fit(self, X: pd.DataFrame, y):
        X = pd.DataFrame(X)
        y = pd.Series(np.asarray(y), index=X.index)
        G, B = (y == 0).sum(), (y == 1).sum()
        self.edges_, self.maps_, iv = {}, {}, {}
        for c in X.columns:
            if pd.api.types.is_numeric_dtype(X[c]):
                v = X[c].dropna().to_numpy(dtype=float)
                e = np.unique(np.quantile(v, np.linspace(0, 1, self.bins + 1))) if len(v) else np.array([])
                e = np.concatenate([[-np.inf], e[1:-1], [np.inf]])
                self.edges_[c] = e
            b = self._bin(X[c], c)
            tab = pd.crosstab(b, y).reindex(columns=[0, 1], fill_value=0)
            g, bad = (tab[0] + 0.5) / G, (tab[1] + 0.5) / B
            woe = np.log(g / bad)
            self.maps_[c] = woe.to_dict()
            iv[c] = float(((g - bad) * woe).sum())
        self.iv_ = pd.Series(iv).sort_values(ascending=False)
        self.columns_ = list(X.columns)
        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=self.columns_)
        out = {c: self._bin(X[c], c).map(self.maps_[c]).fillna(0.0).astype(float) for c in self.columns_}
        return pd.DataFrame(out, index=X.index)

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.columns_)


# ---------------------------------------------------------------- dinh nghia cot
def load_definitions() -> pd.Series:
    """Mo ta cot tu feature_definitions.csv cua cuoc thi (Series: ten cot -> mo ta)."""
    d = get_data_dir()
    cands = [p for p in (d.parent.parent / "feature_definitions.csv", d.parent / "feature_definitions.csv",
                         d / "feature_definitions.csv") if p.exists()]
    if not cands and Path("/kaggle/input").exists():
        cands = [Path(p) for p in glob.glob("/kaggle/input/**/feature_definitions.csv", recursive=True)]
    if not cands:
        print("Khong tim thay feature_definitions.csv - mo ta cot se de trong")
        return pd.Series(dtype=str)
    fd = pd.read_csv(cands[0])
    return fd.set_index(fd.columns[0])[fd.columns[1]].astype(str)


def base_name(col: str) -> str:
    """Ten cot goc: 'applprev_1__maxdpd_577P__mean' -> 'maxdpd_577P'; 'cb__x_1P__max' -> 'x_1P'."""
    parts = col.split("__")
    return parts[1] if len(parts) >= 2 else col


DERIVED_DESC = {
    "ratio_annuity_income": "Derived: monthly annuity / main income (annuity_780A / maininc_215A).",
    "ratio_credit_income": "Derived: credit amount / main income (credamount_770A / maininc_215A).",
    "ratio_annuity_credit": "Derived: monthly annuity / credit amount (annuity_780A / credamount_770A).",
}


def describe(cols, defs: pd.Series) -> pd.Series:
    return pd.Series({c: DERIVED_DESC.get(c) or defs.get(base_name(c), "") for c in cols}, dtype=object)


DUE_WORDS = r"due|maturity|matur|end date|expir|planned|expected|until|next|termination"


def future_date_columns(df: pl.DataFrame, feats, defs: pd.Series, min_share=0.01) -> pd.DataFrame:
    """Cot ngay (da doi thanh so ngay so voi ngay quyet dinh) co gia tri > 0, tuc su kien SAU ngay quyet dinh.

    Giu lai cot co mo ta la ngay den han / ngay ket thuc du kien (biet truoc duoc); cac cot con lai co ty le
    > min_share bi danh dau loai vi co the chua thong tin chi biet sau khi ra quyet dinh.
    """
    import re
    rows = []
    for c in feats:
        b = base_name(c)
        if not b.endswith("D") or not df[c].dtype.is_numeric():
            continue
        share = float((df[c] > 0).fill_null(False).mean())
        if share <= 0:
            continue
        d = defs.get(b, "")
        expected = bool(re.search(DUE_WORDS, d, re.I))
        rows.append({"cot": c, "ty_le_sau_ngay_qd": share, "mo_ta": d,
                     "loai": (share > min_share) and not expected})
    out = pd.DataFrame(rows, columns=["cot", "ty_le_sau_ngay_qd", "mo_ta", "loai"])
    return out.sort_values("ty_le_sau_ngay_qd", ascending=False).reset_index(drop=True)


# ---------------------------------------------------------------- kiem dinh
def gini_stability(df: pd.DataFrame, score_col="score", w_fall=88.0, w_resstd=-0.5) -> dict:
    """Chi so cua cuoc thi Home Credit 2024: mean(Gini tuan) + 88*min(0, do doc) - 0.5*std(phan du)."""
    g = gini_by_week(df, score_col).dropna()
    x = np.arange(len(g))
    a, b = np.polyfit(x, g.values, 1)
    res = g.values - (a * x + b)
    return {"gini_tb": float(g.mean()), "do_doc": float(a), "std_phan_du": float(res.std()),
            "stability": float(g.mean() + w_fall * min(0, a) + w_resstd * res.std())}


def bootstrap_ci(y, p, metric=roc_auc_score, n=500, seed=42, alpha=0.05):
    """Khoang tin cay bootstrap cho mot chi so (mac dinh AUC)."""
    y, p = np.asarray(y), np.asarray(p)
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() == y[i].max():
            continue
        vals.append(metric(y[i], p[i]))
    lo, hi = np.quantile(vals, [alpha / 2, 1 - alpha / 2])
    return float(metric(y, p)), float(lo), float(hi)


def _midrank(x):
    j = np.argsort(x); z = x[j]; n = len(x)
    t = np.zeros(n); i = 0
    while i < n:
        k = i
        while k < n and z[k] == z[i]:
            k += 1
        t[i:k] = 0.5 * (i + k - 1) + 1
        i = k
    out = np.empty(n); out[j] = t
    return out


def delong_test(y, p1, p2) -> dict:
    """Kiem dinh DeLong (DeLong 1988; thuat toan nhanh Sun & Xu 2014) cho chenh lech AUC hai mo hinh tren cung mau."""
    from scipy.stats import norm
    y = np.asarray(y); preds = np.vstack([np.asarray(p1), np.asarray(p2)])
    pos, neg = preds[:, y == 1], preds[:, y == 0]
    m, n = pos.shape[1], neg.shape[1]
    tx = np.array([_midrank(r) for r in pos]); ty = np.array([_midrank(r) for r in neg])
    tz = np.array([_midrank(r) for r in np.hstack([pos, neg])])
    auc = tz[:, :m].sum(axis=1) / (m * n) - (m + 1.0) / (2.0 * n)
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    cov = np.cov(v01) / m + np.cov(v10) / n
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    z = (auc[0] - auc[1]) / np.sqrt(var)
    return {"AUC_1": float(auc[0]), "AUC_2": float(auc[1]), "chenh": float(auc[0] - auc[1]),
            "z": float(z), "p_value": float(2 * norm.sf(abs(z)))}


def calibration_stats(y, p, bins=10) -> dict:
    """ECE (theo thap phan vi), he so do doc va chan cua hieu chinh (logit(y) ~ a + b*logit(p))."""
    from sklearn.linear_model import LogisticRegression
    y, p = np.asarray(y), np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6)
    q = pd.qcut(p, bins, labels=False, duplicates="drop")
    d = pd.DataFrame({"y": y, "p": p, "q": q}).groupby("q").agg(y=("y", "mean"), p=("p", "mean"), n=("y", "size"))
    ece = float((d["n"] * (d["y"] - d["p"]).abs()).sum() / d["n"].sum())
    lg = np.log(p / (1 - p)).reshape(-1, 1)
    lr = LogisticRegression(C=1e6, max_iter=1000).fit(lg, y)
    return {"ECE": ece, "calib_slope": float(lr.coef_[0, 0]), "calib_intercept": float(lr.intercept_[0]),
            "pd_tb": float(p.mean()), "ty_le_thuc": float(y.mean())}


# ---------------------------------------------------------------- nhom hanh vi & credit bureau
import re as _re

BEHAVIOR_GROUPS = {
    "tre_han":       r"days past due|\bdpd\b|overdue|paid late|late payment|past due|delinquen|arrear",
    "vay_nhieu_noi": r"number of (active |open |existing |closed )?(credits?|contracts?|loans?)\b(?! payment)|active (credit|contract)|"
                     r"credit bureau quer|inquir|number of (previous |rejected )?(credit )?applications|"
                     r"applications made by the client|applications that were rejected|financial institution|creditor|lender",
    "ganh_nang_no":  r"debt|outstanding|instal?l?ment amount|annuity|monthly payment|utili[sz]|credit limit",
    "dao_no":        r"refinanc|restructur|prolong|roll.?over|top.?up|early repayment",
}


def assign_group(desc):
    """Nhom hanh vi theo mo ta cot (thu tu uu tien: tre han > vay nhieu noi > no > dao no)."""
    for g, p in BEHAVIOR_GROUPS.items():
        if _re.search(p, desc or "", _re.I):
            return g
    return None


def credit_bureau_features(base: pl.LazyFrame, defs: pd.Series) -> pl.DataFrame:
    """Gop credit_bureau_a_1 ve 1 dong / ho so (chi tu cot that cua bang, khong sinh du lieu):
    so ban ghi; max/mean/sum cot so thuoc nhom hanh vi; so gia tri khac nhau cua to chuc cho vay, muc dich, vai tro;
    so hop dong mo trong 30/90/180 ngay truoc ngay nop ho so va so ngay tu hop dong gan nhat."""
    cb = _with_decision_date("credit_bureau_a_1", base)
    sch = cb.collect_schema()
    desc = {c: defs.get(c, "") for c in sch if c not in KEYS}
    num_cols = [c for c in desc if c[-1] in "PAL" and sch[c].is_numeric() and assign_group(desc[c])]
    cat_cols = [c for c in desc if c[-1] == "M" and _re.search(r"financial institution|creditor|lender|subject|purpose|contract type|type of", desc[c], _re.I)]
    open_cols = [c for c in desc if c[-1] == "D" and _re.search(r"(start|open|begin|conclu|sign)", desc[c], _re.I)
                 and not _re.search(DUE_WORDS, desc[c], _re.I)]
    exprs = [pl.len().alias("cb__so_ban_ghi")]
    for c in num_cols:
        exprs += [pl.col(c).max().alias(f"cb__{c}__max"), pl.col(c).mean().alias(f"cb__{c}__mean"),
                  pl.col(c).sum().alias(f"cb__{c}__sum")]
    for c in cat_cols:
        exprs.append(pl.col(c).drop_nulls().n_unique().alias(f"cb__{c}__so_gia_tri"))
    for c in open_cols:
        for w in (30, 90, 180):
            exprs.append(((pl.col(c) >= -w) & (pl.col(c) <= 0)).sum().alias(f"cb__{c}__mo_trong_{w}n"))
        exprs.append(pl.col(c).filter(pl.col(c) <= 0).max().alias(f"cb__{c}__gan_nhat"))
    out = collect(cb.select(["case_id"] + num_cols + cat_cols + open_cols).group_by("case_id").agg(exprs))
    return out, {"num": num_cols, "cat": cat_cols, "open": open_cols}


def cb_group(col: str, defs: pd.Series):
    if col == "cb__so_ban_ghi" or col.endswith("__so_gia_tri") or "__mo_trong_" in col or col.endswith("__gan_nhat"):
        return "vay_nhieu_noi"
    return assign_group(defs.get(base_name(col), ""))


def align_features(X: pd.DataFrame, feats, schema) -> pd.DataFrame:
    """Dua bang (vd tap test) ve dung danh sach cot va kieu du lieu luc huan luyen (schema polars cua file parquet train)."""
    X = X.copy()
    for c in feats:
        if c not in X.columns:
            X[c] = np.nan
        if schema[c] == pl.Categorical:
            X[c] = X[c].astype("string").astype("category")
        else:
            X[c] = pd.to_numeric(X[c], errors="coerce").astype("float32")
    return X


# ---------------------------------------------------------------- bo nho
def free_memory(ns: dict, min_mb: float = 20.0) -> None:
    """Xoa cac bang du lieu lon con sot lai tu phan truoc trong notebook gop (moi phan tu doc lai du lieu tu file)."""
    import gc
    freed, names = 0.0, []
    for k in list(ns.keys()):
        if k.startswith("_"):
            continue
        v = ns[k]
        try:
            if isinstance(v, pd.DataFrame):
                mb = v.memory_usage(deep=False).sum() / 1e6
            elif isinstance(v, pl.DataFrame):
                mb = v.estimated_size() / 1e6
            elif isinstance(v, np.ndarray):
                mb = v.nbytes / 1e6
            else:
                continue
        except Exception:
            continue
        if mb >= min_mb:
            del ns[k]; freed += mb; names.append(k)
    gc.collect()
    print("Giai phong %.0f MB: %s" % (freed, ", ".join(names) or "(khong co)"))


In [ ]:
# Cai thu vien neu thieu (khi nop bai Internet bi tat - cac thu vien nay co san trong image Kaggle)
import importlib, subprocess, sys
for pkg, mod in [('polars','polars'), ('imbalanced-learn','imblearn'), ('shap','shap')]:
    try: importlib.import_module(mod)
    except ImportError: subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', pkg])

---
# PHẦN 1: 01_EDA

# Đề tài 13 – Đánh giá rủi ro tín dụng cho Mua trước Trả sau (BNPL)
## Notebook 01 – Khám phá dữ liệu (EDA)

**Dữ liệu:** Home Credit – Credit Risk Model Stability (Kaggle, 2024) – dữ liệu thực tế.
**Nguồn:** https://www.kaggle.com/competitions/home-credit-credit-risk-model-stability

Mục tiêu notebook: kiểm chứng kích thước, khoảng thời gian, tỷ lệ vỡ nợ, mức độ thiếu dữ liệu và độ ổn định theo thời gian.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, polars as pl
import matplotlib.pyplot as plt, seaborn as sns
import utils
from utils import META

DATA_DIR, OUT_DIR = utils.get_data_dir(), utils.get_out_dir()
print("Du lieu:", DATA_DIR); print("Ket qua:", OUT_DIR)

## 1. Bảng gốc: kích thước, khoảng thời gian, tỷ lệ vỡ nợ

In [ ]:
base = utils.collect(utils.load_base())
print("So ho so:", base.height)
print("Khoang ngay quyet dinh:", base["date_decision"].min(), "->", base["date_decision"].max())
print("So tuan:", base["WEEK_NUM"].n_unique())
print("Ty le vo no (target=1): %.2f%%" % (100 * base["target"].mean()))
base.head()

> **Ghi vào báo cáo:** số hồ sơ, khoảng ngày (chứng minh yêu cầu *timeliness 7 năm*), tỷ lệ vỡ nợ (mức độ mất cân bằng).

In [ ]:
wk = (base.group_by("WEEK_NUM").agg(pl.len().alias("n"), pl.col("target").mean().alias("bad_rate"))
          .sort("WEEK_NUM").to_pandas())
fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
ax[0].bar(wk["WEEK_NUM"], wk["n"]); ax[0].set_title("So ho so theo tuan"); ax[0].set_xlabel("WEEK_NUM")
ax[1].plot(wk["WEEK_NUM"], 100 * wk["bad_rate"]); ax[1].set_title("Ty le vo no theo tuan (%)"); ax[1].set_xlabel("WEEK_NUM")
plt.tight_layout(); plt.show()

## 2. Tổng quan các bảng sử dụng

In [ ]:
TABLES = ["static_0", "static_cb_0", "applprev_1", "person_1", "credit_bureau_a_1"]
rows = []
for t in TABLES:
    try:
        lf = utils.scan_table(t)
        sch = lf.collect_schema()
        n = lf.select(pl.len()).collect().item()
        n_case = lf.select(pl.col("case_id").n_unique()).collect().item()
        rows.append({"bang": t, "so_file": len(utils.table_files(t)), "so_dong": n, "so_ho_so": n_case,
                     "dong/ho_so": round(n / n_case, 2), "so_cot": len(sch)})
    except FileNotFoundError as e:
        print(e)
pd.DataFrame(rows)

## 3. Giá trị thiếu và quan hệ với biến mục tiêu (bảng `static_0`)

In [ ]:
SAMPLE = 0.2   # lay mau 20% ho so de EDA nhanh; dat None de dung toan bo
b = utils.load_base(SAMPLE)
st = utils.collect(b.join(utils.depth0_features("static_0", b), on="case_id", how="left"))
st = st.join(utils.collect(utils.depth0_features("static_cb_0", b)), on="case_id", how="left")
print(st.shape)

miss = (st.null_count() / st.height).transpose(include_header=True, header_name="cot", column_names=["ty_le_thieu"]).to_pandas()
miss = miss.sort_values("ty_le_thieu", ascending=False)
miss["ty_le_thieu"].plot.hist(bins=20, title="Phan phoi ty le thieu cua cac cot", figsize=(6, 3)); plt.show()
miss.head(15)

In [ ]:
# Information Value (IV) tren mau: xep hang suc phan biet cua tung bien
pdf = utils.prune_features(st).to_pandas()
feats = [c for c in pdf.columns if c not in META]
woe = utils.WoEEncoder().fit(pdf[feats], pdf["target"])
iv = woe.iv_
print("IV > 0.5 can nghi ngo ro ri du lieu:", list(iv[iv > 0.5].index))
iv.head(20).iloc[::-1].plot.barh(figsize=(7, 6), title="Top 20 bien theo IV"); plt.show()

In [ ]:
# Ty le vo no theo bin cua 6 bien so manh nhat
top = [c for c in iv.index if pd.api.types.is_numeric_dtype(pdf[c])][:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, c in zip(axes.ravel(), top):
    q = pd.qcut(pdf[c], 10, duplicates="drop")
    (100 * pdf.groupby(q, observed=True)["target"].mean()).plot.bar(ax=ax)
    ax.set_title(c); ax.set_ylabel("% vo no"); ax.tick_params(axis="x", labelsize=7)
plt.tight_layout(); plt.show()

## 4. Kiểm tra rò rỉ theo thời gian
Cột ngày đã được đổi thành *số ngày so với ngày quyết định*. Giá trị dương = sau ngày quyết định → cần xem ý nghĩa cột (ví dụ ngày đến hạn dự kiến là hợp lệ, ngày trả nợ thực tế thì không).

In [ ]:
date_cols = [c for c in pdf.columns if c.endswith("D") and c not in META]
fut = pd.Series({c: float((pdf[c] > 0).mean()) for c in date_cols}).sort_values(ascending=False)
fut[fut > 0].to_frame("ty_le_gia_tri_sau_ngay_quyet_dinh")

## 5. Nhận xét (tự viết)
- Mức độ mất cân bằng: ...
- Xu hướng tỷ lệ vỡ nợ theo thời gian: ...
- Các biến có sức phân biệt cao và ý nghĩa nghiệp vụ: ...
- Biến nghi ngờ rò rỉ cần loại: ...

---
# PHẦN 2: 02_Feature_Engineering

# Notebook 02 – Xây dựng đặc trưng

Gộp các bảng lịch sử về **một dòng cho mỗi hồ sơ**, tạo đặc trưng hành vi tín dụng, lọc cột và lưu `features_v1.parquet`.

> Bài toán gốc nhắc tới *lịch sử mua sắm và hành vi duyệt web*. Dữ liệu công khai không có hành vi duyệt web; lịch sử các khoản vay/trả góp trước đó (`applprev`, `credit_bureau`) được dùng làm đại diện cho hành vi.

In [ ]:
# Giai phong bo nho con lai tu cac phan truoc (chay TRUOC khi doc du lieu cua phan nay)
import utils
utils.free_memory(globals())

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, polars as pl
import matplotlib.pyplot as plt, seaborn as sns
import utils
from utils import META

DATA_DIR, OUT_DIR = utils.get_data_dir(), utils.get_out_dir()
print("Du lieu:", DATA_DIR); print("Ket qua:", OUT_DIR)

## 1. Cấu hình

In [ ]:
SAMPLE_FRAC = None       # may 8GB RAM: 30% ho so (~450k). None = toan bo (can may manh / Kaggle)
DEPTH0 = ["static_0", "static_cb_0"]
DEPTH1 = ["applprev_1", "person_1"]   # them "credit_bureau_a_1" neu du RAM (bang nang)

## 2. Gộp bảng
Bảng nhiều dòng/hồ sơ được gộp bằng: số bản ghi, max, mean (cột số) và giá trị của bản ghi đầu (cột phân loại).

In [ ]:
%%time
raw = utils.build_features(DEPTH0, DEPTH1, SAMPLE_FRAC)
print(raw.shape)

## 3. Đặc trưng nghiệp vụ bổ sung
Thêm tỷ lệ tự tạo ở đây sau khi đã xem tên cột thật (ví dụ nợ/thu nhập, khoản trả hàng tháng/thu nhập).

In [ ]:
def add_ratio(df, num, den, name):
    if num in df.columns and den in df.columns:
        return df.with_columns((pl.col(num) / pl.when(pl.col(den) > 0).then(pl.col(den))).alias(name))
    print("Bo qua", name, "- thieu cot"); return df

# Ten cot duoi day can doi chieu voi du lieu that (xem feature_definitions.csv cua cuoc thi)
raw = add_ratio(raw, "annuity_780A", "maininc_215A", "ratio_annuity_income")
raw = add_ratio(raw, "credamount_770A", "maininc_215A", "ratio_credit_income")
raw = add_ratio(raw, "annuity_780A", "credamount_770A", "ratio_annuity_credit")

## 4. Lọc cột và ép kiểu

In [ ]:
df = utils.prune_features(raw, max_null=0.95, max_card=200)
feats = [c for c in df.columns if c not in META]
print("Truoc:", raw.width, "cot | Sau:", df.width, "cot | So dac trung:", len(feats))
print("Bo nho: %.1f MB" % (df.estimated_size() / 1e6))

## 4b. Loại cột ngày phát sinh **sau** ngày quyết định (chống rò rỉ)
Cột ngày đã được đổi thành *số ngày so với ngày quyết định*; giá trị dương nghĩa là sự kiện xảy ra sau khi duyệt hồ sơ.
Quy tắc: cột có > 1% giá trị dương bị loại, **trừ** cột mà mô tả trong `feature_definitions.csv` cho biết đó là ngày đến hạn / ngày kết thúc dự kiến (biết trước tại thời điểm duyệt).
Bảng dưới liệt kê toàn bộ cột ngày có giá trị dương cùng quyết định giữ/loại – dùng trực tiếp cho phần phương pháp của báo cáo.

In [ ]:
defs = utils.load_definitions()
fut = utils.future_date_columns(df, feats, defs, min_share=0.01)
fut.to_csv(OUT_DIR / "future_date_check.csv", index=False)
DROP_FUTURE = fut.loc[fut["loai"], "cot"].tolist()
print("So cot ngay co gia tri sau ngay quyet dinh:", len(fut), "| loai:", len(DROP_FUTURE))
pd.set_option("display.max_colwidth", 90)
display(fut.round(4))
df = df.drop(DROP_FUTURE)
feats = [c for c in df.columns if c not in META]
print("So dac trung con lai:", len(feats))

## 4c. Loại biến bị **trôi về độ sẵn có** và thuộc tính nhạy cảm
**Trôi độ sẵn có.** Kiểm định adversarial ở phiên bản trước cho AUC = 1,0: một số cột (ví dụ `description_5085714M`, `contractssum_5085716L`, `requesttype_4525192L`, `birthdate_574D`) gần như **chỉ có dữ liệu ở giai đoạn sau** – ban tổ chức bắt đầu thu thập muộn. Mô hình học trên train (phần lớn rỗng) nhưng gặp giá trị đầy đủ ở giai đoạn sau → hành vi khó lường khi triển khai.
Quy tắc: so sánh tỷ lệ thiếu dữ liệu giữa **train (tuần đầu 70%) và valid (15% kế tiếp)** – không dùng tập test; loại cột có chênh lệch tỷ lệ thiếu ≥ 30 điểm phần trăm.

**Thuộc tính nhạy cảm.** Phân tích độ nhạy (notebook 06, phiên bản trước) cho thấy bỏ giới tính chỉ làm AUC test giảm 0,0019 (p = 0,012) nhưng thu hẹp chênh lệch tỷ lệ duyệt giữa nữ và nam từ 6,3 xuống 1,5 điểm phần trăm. Mô hình cuối vì vậy **không dùng giới tính** (đặt `EXCLUDE_SENSITIVE = False` để tái lập phiên bản cũ).

In [ ]:
EXCLUDE_SENSITIVE, MAX_NULL_SHIFT = True, 0.30
weeks = np.sort(df["WEEK_NUM"].unique()); n_w = len(weeks)
w_tr, w_va = weeks[: int(n_w * 0.70)], weeks[int(n_w * 0.70): int(n_w * 0.85)]
null_tr = df.filter(pl.col("WEEK_NUM").is_in(w_tr.tolist())).select([pl.col(c).is_null().mean().alias(c) for c in feats]).row(0)
null_va = df.filter(pl.col("WEEK_NUM").is_in(w_va.tolist())).select([pl.col(c).is_null().mean().alias(c) for c in feats]).row(0)
shift = pd.DataFrame({"thieu_train": null_tr, "thieu_valid": null_va}, index=feats)
shift["chenh"] = shift["thieu_valid"] - shift["thieu_train"]
shift["mo_ta"] = utils.describe(feats, defs)
DROP_SHIFT = shift.index[shift["chenh"].abs() >= MAX_NULL_SHIFT].tolist()
shift.sort_values("chenh", key=abs, ascending=False).to_csv(OUT_DIR / "availability_shift.csv")
print("Loai do troi do san co (|chenh ty le thieu| >= %.0f%%): %d cot" % (100 * MAX_NULL_SHIFT, len(DROP_SHIFT)))
display(shift.loc[DROP_SHIFT].sort_values("chenh", key=abs, ascending=False).round(3))

import re
SENS = re.compile(r"\bgender\b|\bsex\b", re.I)
DROP_SENS = [c for c in feats if EXCLUDE_SENSITIVE and SENS.search(utils.describe([c], defs).iloc[0] or "")]
print("Loai thuoc tinh nhay cam:", DROP_SENS)
df = df.drop(DROP_SHIFT + DROP_SENS)
feats = [c for c in df.columns if c not in META]
print("So dac trung con lai:", len(feats))

## 5. Xếp hạng đặc trưng theo IV (chỉ fit trên tập train)

In [ ]:
pdf = df.to_pandas()
train, valid, test = utils.time_split(pdf)
print("train/valid/test:", len(train), len(valid), len(test))
tr = train.sample(min(len(train), 300_000), random_state=42)
woe = utils.WoEEncoder().fit(tr[feats], tr["target"])
iv = woe.iv_.rename("IV").to_frame()
iv.to_csv(OUT_DIR / "iv_table.csv")
iv.head(30)

In [ ]:
leak = iv[iv["IV"] > 0.5]
print("Bien co IV > 0.5 (kiem tra ro ri truoc khi giu lai):"); display(leak)
DROP = []          # dien ten cot can loai sau khi kiem tra
df = df.drop(DROP)

## 6. Lưu kết quả

In [ ]:
df.write_parquet(OUT_DIR / "features_v1.parquet")
print("Da luu", OUT_DIR / "features_v1.parquet", df.shape)

---
# PHẦN 3: 03_Modeling_Imbalanced

# Notebook 03 – Mô hình và xử lý dữ liệu mất cân bằng

So sánh **Logistic Regression (WoE)**, **Random Forest**, **LightGBM** với các chiến lược xử lý mất cân bằng: không xử lý, trọng số lớp, undersampling, SMOTE.

Nguyên tắc chống rò rỉ:
- Chia train/valid/test **theo thời gian** (`WEEK_NUM`), test là các tuần cuối.
- Mọi bước có fit (WoE, scaler, SMOTE) nằm trong pipeline, chỉ học từ tập train.
- SMOTE chỉ là kỹ thuật huấn luyện trên tập train; valid/test giữ nguyên dữ liệu thật.
- Chọn mô hình theo tập valid; tập test chỉ dùng để báo cáo.

In [ ]:
# Giai phong bo nho con lai tu cac phan truoc (chay TRUOC khi doc du lieu cua phan nay)
import utils
utils.free_memory(globals())

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, polars as pl
import matplotlib.pyplot as plt, seaborn as sns
import utils
from utils import META

DATA_DIR, OUT_DIR = utils.get_data_dir(), utils.get_out_dir()
print("Du lieu:", DATA_DIR); print("Ket qua:", OUT_DIR)

import joblib, lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
SEED = 42

## 1. Dữ liệu và chia tập

In [ ]:
df = pl.read_parquet(OUT_DIR / "features_v1.parquet").to_pandas()
feats = [c for c in df.columns if c not in META]
train, valid, test = utils.time_split(df)
for n, d in [("train", train), ("valid", valid), ("test", test)]:
    print(f"{n}: {len(d):>9,} ho so | tuan {d.WEEK_NUM.min()}-{d.WEEK_NUM.max()} | vo no {100*d.target.mean():.2f}%")

iv = pd.read_csv(OUT_DIR / "iv_table.csv", index_col=0)["IV"]
TOP_K = 60                                        # so bien cho LR / RF
top_feats = [c for c in iv.index if c in feats][:TOP_K]
N_SUB = 300_000                                   # mau train cho LR / RF (LightGBM dung toan bo)
tr_sub = train.sample(min(len(train), N_SUB), random_state=SEED)

In [ ]:
results, preds = [], {}

def run(name, model, X_tr, y_tr, cols, fit_kw=None):
    model.fit(X_tr[cols], y_tr, **(fit_kw or {}))
    row = {"mo_hinh": name}
    for split, d in [("train", X_tr), ("valid", valid), ("test", test)]:
        p = model.predict_proba(d[cols])[:, 1]
        m = utils.evaluate(d["target"], p)
        row.update({f"{k}_{split}": v for k, v in m.items() if k in ("AUC", "PR_AUC", "KS", "Brier")})
        if split != "train": preds[(name, split)] = p
    results.append(row); print(name, "| AUC valid = %.4f" % row["AUC_valid"])
    return model

## 2. Logistic Regression + WoE (mô hình nền)

In [ ]:
def make_lr(sampler=None, class_weight=None):
    steps = [("woe", utils.WoEEncoder()), ("scale", StandardScaler())]
    if sampler is not None: steps.append(("sampler", sampler))
    steps.append(("clf", LogisticRegression(max_iter=1000, C=0.1, class_weight=class_weight)))
    return Pipeline(steps)

lr_models = {
    "LR | khong xu ly":    make_lr(),
    "LR | class_weight":   make_lr(class_weight="balanced"),
    "LR | undersampling":  make_lr(RandomUnderSampler(sampling_strategy=0.2, random_state=SEED)),
    "LR | SMOTE":          make_lr(SMOTE(sampling_strategy=0.2, random_state=SEED)),
}
for name, m in lr_models.items():
    run(name, m, tr_sub, tr_sub["target"], top_feats)

## 3. Random Forest

In [ ]:
def make_rf(class_weight=None):
    return Pipeline([("woe", utils.WoEEncoder()),
                     ("clf", RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=100,
                                                    max_features="sqrt", class_weight=class_weight,
                                                    n_jobs=-1, random_state=SEED))])

run("RF | khong xu ly", make_rf(), tr_sub, tr_sub["target"], top_feats)
run("RF | class_weight", make_rf("balanced_subsample"), tr_sub, tr_sub["target"], top_feats);

## 4. LightGBM (toàn bộ đặc trưng, xử lý thiếu và biến phân loại trực tiếp)

In [ ]:
LGB_PARAMS = dict(n_estimators=3000, learning_rate=0.03, num_leaves=31, max_depth=-1,
                  min_child_samples=200, subsample=0.8, subsample_freq=1, colsample_bytree=0.6,
                  reg_lambda=5.0, max_bin=255, random_state=SEED, n_jobs=-1, verbose=-1,
                  metric="auc")   # chi theo doi AUC; mac dinh LightGBM con theo doi them binary_logloss
fit_kw = dict(eval_set=[(valid[feats], valid["target"])], eval_metric="auc",
              callbacks=[lgb.early_stopping(200, verbose=False)])
spw = (train["target"] == 0).sum() / (train["target"] == 1).sum()
print("scale_pos_weight day du = %.1f | can bac hai = %.1f" % (spw, np.sqrt(spw)))

lgb_plain = run("LGBM | khong xu ly", lgb.LGBMClassifier(**LGB_PARAMS), train, train["target"], feats, fit_kw)
lgb_spw = run("LGBM | scale_pos_weight", lgb.LGBMClassifier(**LGB_PARAMS, scale_pos_weight=spw), train, train["target"], feats, fit_kw)
lgb_spw_sqrt = run("LGBM | sqrt(scale_pos_weight)", lgb.LGBMClassifier(**LGB_PARAMS, scale_pos_weight=np.sqrt(spw)), train, train["target"], feats, fit_kw)

# So cay thuc su duoc dung sau early stopping: so cay qua it -> mo hinh dung qua som
for n, m in [("khong xu ly", lgb_plain), ("scale_pos_weight", lgb_spw), ("sqrt(scale_pos_weight)", lgb_spw_sqrt)]:
    print(f"{n:<24} best_iteration = {m.best_iteration_}")

**Kiểm tra `scale_pos_weight`.** Lần chạy trước, trọng số đầy đủ (≈ 29) chỉ cho AUC valid 0,72 so với 0,83 của bản không xử lý.
Nguyên nhân nằm ở cấu hình dừng sớm: LightGBM mặc định theo dõi **cả** `binary_logloss` lẫn `auc` trên tập valid và dừng khi *một trong hai* không cải thiện. Với trọng số lớn, xác suất dự đoán bị đẩy lên cao nên logloss trên valid xấu đi gần như ngay từ đầu → mô hình dừng sau rất ít cây, trước khi AUC kịp tăng.
Bản sửa: đặt `metric="auc"` để chỉ dừng theo AUC, nới `early_stopping` lên 200 vòng, in `best_iteration` của từng biến thể và thêm biến thể **căn bậc hai trọng số** (≈ 5,4).
Cách đọc: nếu sau khi sửa ba biến thể có AUC gần nhau thì chênh lệch cũ là do cấu hình chứ không phải do trọng số; khi đó vẫn chọn mô hình theo AUC valid, còn mục tiêu tăng Recall được xử lý bằng **chọn ngưỡng** ở notebook 04 (AUC chỉ phụ thuộc thứ hạng). Brier của bản có trọng số sẽ xấu hơn vì xác suất bị lệch – đã được hiệu chỉnh isotonic ở notebook 04.

### (Tuỳ chọn) Tinh chỉnh bằng Optuna
Đặt `RUN_OPTUNA = True` khi đã có kết quả nền; tinh chỉnh trên mẫu train để tiết kiệm thời gian.

In [ ]:
RUN_OPTUNA, N_TRIALS = False, 30
if RUN_OPTUNA:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    def objective(t):
        p = dict(LGB_PARAMS, num_leaves=t.suggest_int("num_leaves", 15, 127),
                 min_child_samples=t.suggest_int("min_child_samples", 50, 500),
                 colsample_bytree=t.suggest_float("colsample_bytree", 0.3, 0.9),
                 reg_lambda=t.suggest_float("reg_lambda", 0.1, 20, log=True), learning_rate=0.05)
        m = lgb.LGBMClassifier(**p).fit(tr_sub[feats], tr_sub["target"], **fit_kw)
        return utils.evaluate(valid["target"], m.predict_proba(valid[feats])[:, 1])["AUC"]
    study = optuna.create_study(direction="maximize"); study.optimize(objective, n_trials=N_TRIALS)
    print(study.best_value, study.best_params)
    lgb_tuned = run("LGBM | Optuna", lgb.LGBMClassifier(**dict(LGB_PARAMS, **study.best_params)),
                    train, train["target"], feats, fit_kw)

## 5. So sánh kết quả

In [ ]:
res = pd.DataFrame(results).set_index("mo_hinh")
res["chenh_AUC_train_test"] = res["AUC_train"] - res["AUC_test"]
res.to_csv(OUT_DIR / "model_comparison.csv")
res[["AUC_train", "AUC_valid", "AUC_test", "PR_AUC_test", "KS_test", "Brier_test", "chenh_AUC_train_test"]].round(4)

**Cách đọc bảng**
- Xử lý mất cân bằng thường **không** làm tăng AUC nhiều; tác dụng chính là thay đổi Recall ở một ngưỡng và làm **lệch xác suất** (Brier xấu đi) → cần hiệu chỉnh ở notebook 04.
- `chenh_AUC_train_test` > 0,05: dấu hiệu overfit. AUC test > 0,9: nghi ngờ rò rỉ dữ liệu.

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for name in ["LR | class_weight", "RF | class_weight", "LGBM | khong xu ly", "LGBM | scale_pos_weight", "LGBM | sqrt(scale_pos_weight)"]:
    p = preds[(name, "test")]
    fpr, tpr, _ = roc_curve(test["target"], p); ax[0].plot(fpr, tpr, label=name)
    pr, rc, _ = precision_recall_curve(test["target"], p); ax[1].plot(rc, pr, label=name)
ax[0].plot([0, 1], [0, 1], "k--"); ax[0].set_title("ROC (test)"); ax[0].legend()
ax[1].set_title("Precision-Recall (test)"); ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision")
plt.tight_layout(); plt.show()

## 6. Lưu mô hình tốt nhất (chọn theo AUC valid) và dự đoán cho notebook 04

In [ ]:
lgb_models = {"LGBM | khong xu ly": lgb_plain, "LGBM | scale_pos_weight": lgb_spw,
              "LGBM | sqrt(scale_pos_weight)": lgb_spw_sqrt}
if RUN_OPTUNA: lgb_models["LGBM | Optuna"] = lgb_tuned
best_name = res.loc[list(lgb_models), "AUC_valid"].idxmax()
best = lgb_models[best_name]; print("Mo hinh chon:", best_name)

joblib.dump({"model": best, "features": feats, "name": best_name}, OUT_DIR / "best_model.joblib")
for split, d in [("valid", valid), ("test", test)]:
    out = d[["case_id", "WEEK_NUM", "target"]].copy(); out["pd_raw"] = preds[(best_name, split)]
    out.to_parquet(OUT_DIR / f"preds_{split}.parquet", index=False)
# Luu du doan cua moi mo hinh (dung cho kiem dinh DeLong o notebook 06)
for split, d in [("valid", valid), ("test", test)]:
    allp = d[["case_id", "WEEK_NUM", "target"]].copy()
    for (name, sp), p in preds.items():
        if sp == split: allp[name] = p
    allp.to_parquet(OUT_DIR / f"preds_all_{split}.parquet", index=False)
train[["case_id"]].assign(pd_raw=best.predict_proba(train[feats])[:, 1]).to_parquet(OUT_DIR / "preds_train.parquet", index=False)

imp = pd.Series(best.booster_.feature_importance("gain"), index=feats).sort_values(ascending=False)
(imp / imp.sum()).head(20).iloc[::-1].plot.barh(figsize=(7, 6), title="Top 20 dac trung (gain)"); plt.show()
print("Ty trong dac trung lon nhat: %.1f%% (qua lon -> kiem tra ro ri)" % (100 * imp.iloc[0] / imp.sum()))

---
# PHẦN 4: 04_Scorecard_Credit_Limit

# Notebook 04 – Điểm tín dụng, hạn mức BNPL và giải thích mô hình

1. Hiệu chỉnh xác suất vỡ nợ (PD)
2. Quy đổi PD → điểm tín dụng → hạng rủi ro
3. Chính sách duyệt và hạn mức tự động
4. Phân tích lợi nhuận theo ngưỡng duyệt
5. Độ ổn định theo thời gian (Gini theo tuần, PSI)
6. Giải thích bằng SHAP

Ngưỡng và hạng rủi ro được xác định trên tập **valid**; tập **test** chỉ dùng để báo cáo.

In [ ]:
# Giai phong bo nho con lai tu cac phan truoc (chay TRUOC khi doc du lieu cua phan nay)
import utils
utils.free_memory(globals())

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, polars as pl
import matplotlib.pyplot as plt, seaborn as sns
import utils
from utils import META

DATA_DIR, OUT_DIR = utils.get_data_dir(), utils.get_out_dir()
print("Du lieu:", DATA_DIR); print("Ket qua:", OUT_DIR)

import joblib
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import calibration_curve

va = pd.read_parquet(OUT_DIR / "preds_valid.parquet")
te = pd.read_parquet(OUT_DIR / "preds_test.parquet")
tr = pd.read_parquet(OUT_DIR / "preds_train.parquet")

## 1. Hiệu chỉnh xác suất (isotonic, fit trên valid)

In [ ]:
iso = IsotonicRegression(y_min=1e-4, y_max=1 - 1e-4, out_of_bounds="clip").fit(va["pd_raw"], va["target"])
for d in (va, te): d["pd"] = iso.predict(d["pd_raw"])

print("Test - truoc hieu chinh:", {k: round(v, 4) for k, v in utils.evaluate(te.target, te.pd_raw).items()})
print("Test - sau hieu chinh  :", {k: round(v, 4) for k, v in utils.evaluate(te.target, te.pd).items()})
print("PD trung binh du doan %.2f%% | ty le vo no thuc te %.2f%%" % (100 * te.pd.mean(), 100 * te.target.mean()))

fig, ax = plt.subplots(figsize=(5, 4.5))
for col, lab in [("pd_raw", "truoc"), ("pd", "sau")]:
    y, x = calibration_curve(te["target"], te[col], n_bins=10, strategy="quantile"); ax.plot(x, y, "o-", label=lab)
m = max(ax.get_xlim()[1], ax.get_ylim()[1]); ax.plot([0, m], [0, m], "k--")
ax.set_xlabel("PD du doan"); ax.set_ylabel("Ty le vo no thuc te"); ax.set_title("Calibration (test)"); ax.legend(); plt.show()

## 2. Điểm tín dụng và hạng rủi ro
`score = OFFSET + FACTOR × ln(odds)` với odds = (1 − PD) / PD. Quy ước: 600 điểm tại odds 50:1, cứ +20 điểm thì odds gấp đôi (PDO = 20).

In [ ]:
PDO, BASE_SCORE, BASE_ODDS = 20, 600, 50
FACTOR = PDO / np.log(2); OFFSET = BASE_SCORE - FACTOR * np.log(BASE_ODDS)
for d in (va, te): d["score"] = OFFSET + FACTOR * np.log((1 - d["pd"]) / d["pd"])

BANDS = ["E", "D", "C", "B", "A"]                       # E = rui ro cao nhat
edges = np.quantile(va["score"], [0.10, 0.30, 0.60, 0.85])   # xac dinh tren valid
edges = np.concatenate([[-np.inf], np.unique(edges), [np.inf]])
labels = BANDS[-(len(edges) - 1):]
for d in (va, te): d["band"] = pd.cut(d["score"], edges, labels=labels)

band_tbl = te.groupby("band", observed=True).agg(so_ho_so=("target", "size"), ty_le_vo_no=("target", "mean"),
                                                 pd_tb=("pd", "mean"), diem_min=("score", "min"), diem_max=("score", "max"))
band_tbl["ty_trong"] = band_tbl["so_ho_so"] / band_tbl["so_ho_so"].sum()
band_tbl.round(4)

> Kiểm tra: tỷ lệ vỡ nợ phải **giảm dần** từ hạng E đến hạng A trên tập test. Nếu không đơn điệu, gộp hạng hoặc đổi điểm cắt.

## 3. Chính sách duyệt và hạn mức

Các tham số dưới đây là **giả định nghiệp vụ** để minh hoạ (không phải dữ liệu) – cần nêu rõ trong báo cáo và có thể điều chỉnh.

In [ ]:
POLICY = pd.DataFrame({
    "band":      ["A", "B", "C", "D", "E"],
    "quyet_dinh": ["Duyet", "Duyet", "Duyet", "Duyet han che", "Tu choi"],
    "han_muc":   [20_000_000, 12_000_000, 6_000_000, 2_000_000, 0],   # VND, gia dinh
}).set_index("band")
te["han_muc"] = te["band"].astype(str).map(POLICY["han_muc"]).fillna(0)
te["quyet_dinh"] = te["band"].astype(str).map(POLICY["quyet_dinh"])
display(POLICY)
print("Ty le duyet: %.1f%%" % (100 * (te.han_muc > 0).mean()))
print("Ty le vo no trong nhom duoc duyet: %.2f%% (toan bo: %.2f%%)" % (100 * te.loc[te.han_muc > 0, "target"].mean(), 100 * te.target.mean()))

## 4. Lợi nhuận theo chính sách hạn mức và theo ngưỡng duyệt
Giả định nghiệp vụ: khoản không vỡ nợ mang lại `MARGIN` × hạn mức; khoản vỡ nợ mất `LGD` × hạn mức (dư nợ tại thời điểm vỡ nợ = hạn mức, giả định thận trọng).
- **4a.** Lợi nhuận của chính sách ở mục 3, tính đúng hạn mức của từng hạng.
- **4b.** Đường cong lợi nhuận theo ngưỡng PD: hồ sơ được duyệt nhận hạn mức của hạng mình (hạng E nếu được duyệt nhận hạn mức thấp nhất 2 triệu). Ngưỡng tối ưu chọn trên valid.
- **4c.** Phân tích độ nhạy: cùng đường cong nhưng mọi hồ sơ cùng hạn mức 6 triệu (cách tính của phiên bản trước) – chỉ để so sánh.

In [ ]:
MARGIN, LGD = 0.06, 0.60          # gia dinh nghiep vu
LIM_IF_APPROVED = POLICY["han_muc"].replace(0, POLICY.loc[POLICY["han_muc"] > 0, "han_muc"].min())  # E -> 2tr neu duoc duyet

def pnl(target, limit):
    return np.where(np.asarray(target) == 0, MARGIN, -LGD) * np.asarray(limit, dtype=float)

# ---- 4a. Chinh sach theo hang (test)
te["pnl"] = pnl(te["target"], te["han_muc"])
te["el"] = te["pd"] * LGD * te["han_muc"]                     # ton that ky vong theo PD da hieu chinh
pol = te.groupby("band", observed=True).agg(so_ho_so=("target", "size"), ty_le_vo_no=("target", "mean"),
                                            han_muc=("han_muc", "first"), tong_han_muc=("han_muc", "sum"),
                                            ton_that_ky_vong=("el", "sum"), loi_nhuan=("pnl", "sum"))
pol.loc["Tong"] = [pol["so_ho_so"].sum(), te["target"].mean(), np.nan, pol["tong_han_muc"].sum(),
                   pol["ton_that_ky_vong"].sum(), pol["loi_nhuan"].sum()]
for c in ["tong_han_muc", "ton_that_ky_vong", "loi_nhuan"]: pol[c] = pol[c] / 1e9     # ty VND
display(pol.round(4))
print("Loi nhuan chinh sach theo hang (test): %.1f ty VND tren tong han muc %.1f ty" % (pol.loc["Tong", "loi_nhuan"], pol.loc["Tong", "tong_han_muc"]))

# ---- 4b. Duong cong theo nguong PD, han muc theo hang
def profit_curve(d, limit=None):
    lim = (d["band"].astype(str).map(LIM_IF_APPROVED).astype(float) if limit is None
           else pd.Series(float(limit), index=d.index))
    rows = []
    for t in np.quantile(d["pd"], np.linspace(0.05, 1, 40)):
        ok = d["pd"] <= t
        rows.append({"nguong_pd": t, "ty_le_duyet": ok.mean(), "ty_le_vo_no": d.loc[ok, "target"].mean(),
                     "loi_nhuan": pnl(d.loc[ok, "target"], lim[ok]).sum()})
    return pd.DataFrame(rows)

def report(pc_va, d, limit=None, label=""):
    t = pc_va.loc[pc_va["loi_nhuan"].idxmax(), "nguong_pd"]         # chon tren valid
    lim = (d["band"].astype(str).map(LIM_IF_APPROVED).astype(float) if limit is None
           else pd.Series(float(limit), index=d.index))
    ok = d["pd"] <= t
    print("%s nguong PD toi uu (valid) %.3f -> test: duyet %.1f%% | vo no %.2f%% | loi nhuan %.1f ty (duyet tat ca: %.1f ty)" % (
        label, t, 100 * ok.mean(), 100 * d.loc[ok, "target"].mean(),
        pnl(d.loc[ok, "target"], lim[ok]).sum() / 1e9, pnl(d["target"], lim).sum() / 1e9))
    return t

pc_va, pc_te = profit_curve(va), profit_curve(te)
best_t = report(pc_va, te, label="[Han muc theo hang]")

# ---- 4c. Do nhay: han muc co dinh 6tr
pc_va6, pc_te6 = profit_curve(va, 6_000_000), profit_curve(te, 6_000_000)
report(pc_va6, te, 6_000_000, label="[Han muc co dinh 6tr]")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(100 * pc_te["ty_le_duyet"], pc_te["loi_nhuan"] / 1e9, "o-", label="han muc theo hang")
ax.plot(100 * pc_te6["ty_le_duyet"], pc_te6["loi_nhuan"] / 1e9, "s--", label="co dinh 6 trieu (do nhay)")
ax.set_xlabel("Ty le duyet (%)"); ax.set_ylabel("Loi nhuan (ty VND)"); ax.set_title("Loi nhuan theo ty le duyet (test)")
ax.legend(); plt.show()

## 5. Độ ổn định theo thời gian

In [ ]:
gw = utils.gini_by_week(te.assign(score=te["pd"]))
print("Gini theo tuan: tb %.3f | min %.3f | do lech chuan %.3f" % (gw.mean(), gw.min(), gw.std()))
gw.plot(marker="o", figsize=(8, 3), title="Gini theo tuan (test)"); plt.axhline(gw.mean(), ls="--", c="gray"); plt.show()
print("PSI diem du doan train -> test: %.3f  (>0.25 la troi manh)" % utils.psi(tr["pd_raw"], te["pd_raw"]))

## 6. Giải thích mô hình bằng SHAP

In [ ]:
import shap
pack = joblib.load(OUT_DIR / "best_model.joblib"); model, feats = pack["model"], pack["features"]
df = pl.read_parquet(OUT_DIR / "features_v1.parquet").to_pandas()
_, _, test_full = utils.time_split(df)
Xs = test_full.sample(min(len(test_full), 5000), random_state=42)

explainer = shap.TreeExplainer(model.booster_)
sv = explainer.shap_values(Xs[feats])
sv = sv[1] if isinstance(sv, list) else sv
shap.summary_plot(sv, Xs[feats], max_display=20)

In [ ]:
# Ly do tu choi cho mot ho so rui ro cao: 5 dac trung day PD len nhieu nhat
i = int(np.argmax(sv.sum(axis=1)))
reason = pd.DataFrame({"gia_tri": Xs[feats].iloc[i].astype(object).values, "shap": sv[i]}, index=feats)
print("case_id:", Xs["case_id"].iloc[i], "| target thuc te:", Xs["target"].iloc[i])
reason.sort_values("shap", ascending=False).head(5)

## 7. Kết luận (tự viết)
- Mô hình chọn, chỉ số trên test, mức ổn định theo thời gian: ...
- Chính sách hạn mức đề xuất và tác động (tỷ lệ duyệt, tỷ lệ vỡ nợ, lợi nhuận): ...
- Hạn chế: không có dữ liệu hành vi duyệt web; tham số lợi nhuận là giả định; ...

---
# PHẦN 5: 05_Early_Warning_Behavior

# Notebook 05 – Tín hiệu cảnh báo sớm: hành vi trước khi vỡ nợ

Ngoài xác suất vỡ nợ, notebook này tìm các **hành vi thường xuất hiện trước nợ xấu** có thể đo được từ dữ liệu hiện có:

| Nhóm hành vi | Ý nghĩa nghiệp vụ |
|---|---|
| **Trễ hạn thanh toán** | số ngày quá hạn (DPD), tỷ lệ kỳ trả muộn, số kỳ có DPD trong lịch sử |
| **Vay nhiều nơi** (loan stacking) | nhiều hợp đồng đang hoạt động, nhiều tổ chức cho vay, nhiều lần tra cứu/nộp hồ sơ gần đây |
| **Gánh nặng nợ** | dư nợ, khoản trả hàng kỳ, tỷ lệ trên thu nhập |
| **Vay mới để trả nợ cũ** (đảo nợ) | mở hợp đồng mới ngay trước khi nộp hồ sơ, hồ sơ vay dồn dập, các trường tái cơ cấu / gia hạn nếu có |

**Lưu ý phương pháp.** Dữ liệu chỉ có một nhãn là vỡ nợ (`target`); không có nhãn "sẽ trễ hạn" hay "sẽ đảo nợ" trong tương lai. Vì vậy các hành vi trên được dùng như **chỉ báo dẫn trước** đo tại thời điểm nộp hồ sơ (chỉ dùng thông tin trước ngày quyết định), và được đánh giá bằng mức độ chúng báo trước vỡ nợ trên tập test.

Nhóm biến được chọn **tự động theo mô tả cột** trong `feature_definitions.csv` (không đoán theo tên cột) và in ra để kiểm tra lại.

In [ ]:
# Giai phong bo nho con lai tu cac phan truoc (chay TRUOC khi doc du lieu cua phan nay)
import utils
utils.free_memory(globals())

In [ ]:
import warnings, re; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, polars as pl
import matplotlib.pyplot as plt
import utils
from utils import META
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

DATA_DIR, OUT_DIR = utils.get_data_dir(), utils.get_out_dir()
SEED = 42
pd.set_option("display.max_colwidth", 90)
defs = utils.load_definitions()
print("So cot co mo ta:", len(defs))

## 1. Định nghĩa nhóm hành vi theo mô tả cột

In [ ]:
GROUPS = utils.BEHAVIOR_GROUPS               # regex theo mo ta cot, dinh nghia trong utils.py
assign_group = utils.assign_group
GROUP_VN = {"tre_han": "Tre han", "vay_nhieu_noi": "Vay nhieu noi", "ganh_nang_no": "Ganh nang no", "dao_no": "Dao no / tai co cau"}
for g, p in GROUPS.items(): print(f"{GROUP_VN[g]:<22} {p}")

df = pl.read_parquet(OUT_DIR / "features_v1.parquet")
feats = [c for c in df.columns if c not in META]
desc = utils.describe(feats, defs)
grp = desc.map(assign_group)
print(grp.value_counts(dropna=False).rename("so_bien"))

In [ ]:
iv = pd.read_csv(OUT_DIR / "iv_table.csv", index_col=0)["IV"]
tab = pd.DataFrame({"nhom": grp, "IV": iv.reindex(feats), "mo_ta": desc}).dropna(subset=["nhom"])
for g in GROUPS:
    t = tab[tab["nhom"] == g].sort_values("IV", ascending=False)
    print(f"\n=== {GROUP_VN[g]}: {len(t)} bien | 8 bien IV cao nhat"); display(t.head(8).round(4))

## 2. Bổ sung đặc trưng hành vi từ lịch sử tín dụng bên ngoài (`credit_bureau_a_1`)
Bảng này (≈ 16 triệu dòng, nhiều hợp đồng/hồ sơ) chưa được dùng ở notebook 02 vì nặng. Ở đây chỉ lấy các cột liên quan đến bốn nhóm hành vi và gộp thành:
- số hợp đồng, **số tổ chức cho vay khác nhau** (vay nhiều nơi),
- DPD / số tiền quá hạn lớn nhất, trung bình (trễ hạn),
- tổng dư nợ, tổng khoản trả hàng kỳ (gánh nặng nợ),
- **số hợp đồng mở trong 30/90/180 ngày trước ngày nộp hồ sơ** (dấu hiệu vay mới dồn dập / vay để trả nợ cũ).

In [ ]:
base = utils.load_base()
cbf, cb_info = utils.credit_bureau_features(base, defs)          # ham dung chung voi notebook 07 (tap test)
print("Cot so:", len(cb_info["num"]), "| cot phan loai:", cb_info["cat"], "| cot ngay mo hop dong:", cb_info["open"])
display(pd.Series({c: defs.get(c, "") for c in cb_info["num"] + cb_info["cat"] + cb_info["open"]}, name="mo_ta").to_frame())
cbf = utils.prune_features(cbf, max_null=0.95)
print("Dac trung moi tu credit_bureau_a_1:", cbf.width - 1)

# Cung quy tac troi do san co nhu notebook 02 (train vs valid, khong dung test)
weeks = np.sort(df["WEEK_NUM"].unique()); n_w = len(weeks)
wk = df.select("case_id", "WEEK_NUM")
cbw = cbf.join(wk, on="case_id", how="right")
w_tr, w_va = weeks[: int(n_w * 0.70)].tolist(), weeks[int(n_w * 0.70): int(n_w * 0.85)].tolist()
cols = [c for c in cbf.columns if c != "case_id"]
n_tr = cbw.filter(pl.col("WEEK_NUM").is_in(w_tr)).select([pl.col(c).is_null().mean().alias(c) for c in cols]).row(0)
n_va = cbw.filter(pl.col("WEEK_NUM").is_in(w_va)).select([pl.col(c).is_null().mean().alias(c) for c in cols]).row(0)
drop_cb = [c for c, a, b in zip(cols, n_tr, n_va) if abs(b - a) >= 0.30]
cbf = cbf.drop(drop_cb)
print("Loai do troi do san co:", len(drop_cb), "| con lai:", cbf.width - 1)

In [ ]:
# Gan nhom hanh vi cho dac trung moi
new_feats = [c for c in cbf.columns if c != "case_id"]
grp_new = pd.Series({c: utils.cb_group(c, defs) for c in new_feats})
grp_all = pd.concat([grp, grp_new])

dfx = df.join(cbf, on="case_id", how="left")
pdf = dfx.to_pandas()
train, valid, test = utils.time_split(pdf)
feats_all = feats + new_feats
print("Tong dac trung:", len(feats_all), "| train/valid/test:", len(train), len(valid), len(test))
print(grp_new.value_counts(dropna=False).rename("so_bien_moi"))

## 3. Chỉ số cảnh báo sớm theo từng nhóm hành vi
Mỗi nhóm được tóm thành một **điểm hành vi**: Logistic Regression trên biến WoE của riêng nhóm đó (fit trên train).
Cờ cảnh báo của nhóm bật khi điểm thuộc **20% rủi ro nhất** (ngưỡng cố định từ train). Đánh giá trên test:
độ phủ (% hồ sơ bị gắn cờ), tỷ lệ vỡ nợ khi có / không có cờ, **lift** (= tỷ lệ vỡ nợ khi có cờ ÷ tỷ lệ chung) và AUC của riêng điểm nhóm.

In [ ]:
tr_s = train.sample(min(len(train), 300_000), random_state=SEED)
base_rate = test["target"].mean()
scores, rows = {}, []
for g in GROUPS:
    cols = [c for c in feats_all if grp_all.get(c) == g]
    if not cols:
        print("Bo qua nhom", g, "- khong co bien"); continue
    m = Pipeline([("woe", utils.WoEEncoder()), ("sc", StandardScaler()),
                  ("lr", LogisticRegression(max_iter=1000, C=0.1))]).fit(tr_s[cols], tr_s["target"])
    s_tr = m.predict_proba(tr_s[cols])[:, 1]; thr = np.quantile(s_tr, 0.80)
    s_te = m.predict_proba(test[cols])[:, 1]
    # nhieu ho so cung diem (bien thua, phan lon rong) -> dung '>' de co khong bat cho ca nhom diem trung
    flag = (s_te >= thr) if (s_tr >= thr).mean() <= 0.30 else (s_te > thr)
    scores[g] = s_te; test["co_" + g] = flag.astype(int)
    rows.append({"nhom": GROUP_VN[g], "so_bien": len(cols), "AUC_test": roc_auc_score(test["target"], s_te),
                 "do_phu": flag.mean(), "vo_no_khi_co_co": test.loc[flag, "target"].mean(),
                 "vo_no_khi_khong": test.loc[~flag, "target"].mean(),
                 "lift": test.loc[flag, "target"].mean() / base_rate})
ewi = pd.DataFrame(rows).set_index("nhom")
ewi.to_csv(OUT_DIR / "early_warning_groups.csv")
print("Ty le vo no chung (test): %.2f%%" % (100 * base_rate)); ewi.round(4)

## 4. Hồ sơ hành vi: số cờ cảnh báo và tổ hợp hành vi
Nếu các hành vi thực sự là tín hiệu dẫn trước, tỷ lệ vỡ nợ phải **tăng dần theo số cờ** đồng thời bật.

In [ ]:
flag_cols = [c for c in test.columns if c.startswith("co_")]
test["so_co"] = test[flag_cols].sum(axis=1)
prof = test.groupby("so_co").agg(so_ho_so=("target", "size"), ty_le_vo_no=("target", "mean"))
prof["ty_trong"] = prof["so_ho_so"] / len(test); prof["lift"] = prof["ty_le_vo_no"] / base_rate
display(prof.round(4))

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(prof.index.astype(str), 100 * prof["ty_le_vo_no"]); ax.axhline(100 * base_rate, ls="--", c="gray")
ax.set_xlabel("So co canh bao dong thoi"); ax.set_ylabel("% vo no (test)"); ax.set_title("Ty le vo no theo so hanh vi rui ro"); plt.show()

combo = test[flag_cols].astype(str).agg("".join, axis=1).map(
    lambda s: " + ".join(GROUP_VN[c[3:]] for c, v in zip(flag_cols, s) if v == "1") or "(khong co co)")
cmb = test.assign(to_hop=combo).groupby("to_hop").agg(so_ho_so=("target", "size"), ty_le_vo_no=("target", "mean"))
cmb = cmb[cmb["so_ho_so"] >= 300].sort_values("ty_le_vo_no", ascending=False)
cmb["lift"] = cmb["ty_le_vo_no"] / base_rate
cmb.to_csv(OUT_DIR / "early_warning_combos.csv"); cmb.head(12).round(4)

## 5. Đặc trưng hành vi bổ sung có cải thiện mô hình vỡ nợ không?
So sánh LightGBM trên bộ đặc trưng gốc và bộ có thêm đặc trưng `credit_bureau_a_1`, cùng tham số và cùng cách chia tập.

In [ ]:
P = dict(n_estimators=3000, learning_rate=0.03, num_leaves=31, min_child_samples=200, subsample=0.8, subsample_freq=1,
         colsample_bytree=0.6, reg_lambda=5.0, random_state=SEED, n_jobs=-1, verbose=-1, metric="auc")
res, models = [], {}
for name, cols in [("Dac trung goc", feats), ("Goc + hanh vi credit bureau", feats_all)]:
    m = lgb.LGBMClassifier(**P).fit(train[cols], train["target"], eval_set=[(valid[cols], valid["target"])],
                                     callbacks=[lgb.early_stopping(200, verbose=False)])
    p_te = m.predict_proba(test[cols])[:, 1]
    gw = utils.gini_by_week(test.assign(score=p_te))
    res.append({"bo_dac_trung": name, "so_bien": len(cols), "best_iteration": m.best_iteration_,
                "AUC_valid": roc_auc_score(valid["target"], m.predict_proba(valid[cols])[:, 1]),
                "AUC_test": roc_auc_score(test["target"], p_te), "Gini_tuan_tb": gw.mean(), "Gini_tuan_min": gw.min()})
    models[name] = (m, cols)
res = pd.DataFrame(res).set_index("bo_dac_trung"); res.to_csv(OUT_DIR / "behavior_feature_gain.csv"); display(res.round(4))

# Kiem dinh DeLong: chenh lech AUC test giua hai bo dac trung co y nghia thong ke khong
p_base = models["Dac trung goc"][0].predict_proba(test[feats])[:, 1]
p_full = models["Goc + hanh vi credit bureau"][0].predict_proba(test[feats_all])[:, 1]
dl = utils.delong_test(test["target"], p_full, p_base)
print("DeLong (goc + credit bureau vs goc): chenh AUC = %.4f | z = %.2f | p = %.2e" % (dl["chenh"], dl["z"], dl["p_value"]))

# Luu bo dac trung mo rong, mo hinh cuoi va du doan cho notebook 06-07
import joblib
FINAL = "LGBM | goc + credit bureau"
fm, fcols = models["Goc + hanh vi credit bureau"]
joblib.dump({"model": fm, "features": fcols, "name": FINAL}, OUT_DIR / "final_model.joblib")
for split, d in [("valid", valid), ("test", test)]:
    p = fm.predict_proba(d[fcols])[:, 1]
    d[["case_id", "WEEK_NUM", "target"]].assign(pd_raw=p).to_parquet(OUT_DIR / f"preds_final_{split}.parquet", index=False)
    allp = pd.read_parquet(OUT_DIR / f"preds_all_{split}.parquet")
    allp = allp.merge(d[["case_id"]].assign(**{FINAL: p}), on="case_id", how="left")
    allp.to_parquet(OUT_DIR / f"preds_all_{split}.parquet", index=False)
dfx.write_parquet(OUT_DIR / "features_v2.parquet")
pd.Series(new_feats, name="cot").to_csv(OUT_DIR / "features_v2_new_cols.csv", index=False)
grp_all.rename("nhom").to_csv(OUT_DIR / "behavior_groups.csv")
print("Da luu features_v2.parquet", dfx.shape)

In [ ]:
# Dong gop (gain) cua tung nhom hanh vi trong mo hinh day du
m, cols = models["Goc + hanh vi credit bureau"]
gain = pd.Series(m.booster_.feature_importance("gain"), index=cols)
share = (gain.groupby(grp_all.reindex(cols).fillna("khac").map(lambda g: GROUP_VN.get(g, "Khac"))).sum() / gain.sum()).sort_values()
share.plot.barh(figsize=(6, 3), title="Ty trong gain theo nhom hanh vi"); plt.show()
print((100 * share).round(1).astype(str) + "%")
print("\nTop 15 dac trung:"); display(pd.DataFrame({"gain_%": 100 * gain / gain.sum(), "nhom": grp_all.reindex(cols),
                                                     "mo_ta": utils.describe(cols, defs)}).sort_values("gain_%", ascending=False).head(15).round(2))

## 6. Nhận xét (tự viết)
- Nhóm hành vi báo trước vỡ nợ mạnh nhất (lift, AUC): ...
- Tỷ lệ vỡ nợ theo số cờ cảnh báo – có tăng đơn điệu không: ...
- Tổ hợp hành vi nguy hiểm nhất (ví dụ trễ hạn + vay nhiều nơi): ...
- Đặc trưng từ lịch sử tín dụng bên ngoài có cải thiện AUC / độ ổn định không: ...
- Ứng dụng: dùng cờ cảnh báo để **giám sát sau giải ngân** (theo dõi lại hồ sơ khi cờ bật), giảm hạn mức hoặc yêu cầu xác minh thêm.
- Hạn chế: không có nhãn trễ hạn/đảo nợ trong tương lai; nhóm biến chọn theo từ khoá mô tả nên cần rà soát bảng ở mục 1.

---
# PHẦN 6: 06_Validation

# Notebook 06 – Kiểm định mô hình

Notebook này **không tạo đặc trưng mới**; chỉ kiểm định các mô hình đã xây ở notebook 03 và 05 theo các tiêu chí học thuật và ứng dụng:

1. **Độ chính xác có ý nghĩa thống kê**: khoảng tin cậy bootstrap 95% của AUC/Gini; kiểm định DeLong cho chênh lệch AUC giữa các mô hình.
2. **Độ ổn định theo thời gian**: chỉ số ổn định Gini của cuộc thi (trung bình Gini tuần + phạt xu hướng giảm + phạt dao động).
3. **Kiểm định cửa sổ thời gian** (rolling-origin): cửa sổ mở rộng và cửa sổ trượt, 4 khối kiểm định liên tiếp, có khoảng đệm.
4. **Hiệu chỉnh xác suất**: ECE, hệ số độ dốc / chặn của hiệu chỉnh trước và sau isotonic.
5. **Trôi dữ liệu**: adversarial validation (phân biệt train với test) và PSI từng biến.
6. **Kiểm tra rò rỉ của quy trình**: huấn luyện với nhãn bị hoán vị ngẫu nhiên – AUC phải ≈ 0,5.

Tập test (tuần 78–91) chỉ dùng để báo cáo, không dùng để chọn tham số.

In [ ]:
# Giai phong bo nho con lai tu cac phan truoc (chay TRUOC khi doc du lieu cua phan nay)
import utils
utils.free_memory(globals())

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, polars as pl
import matplotlib.pyplot as plt
import utils
from utils import META
import lightgbm as lgb
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score

DATA_DIR, OUT_DIR = utils.get_data_dir(), utils.get_out_dir()
SEED = 42
summary = {}          # bang tong hop cuoi notebook

pv = pd.read_parquet(OUT_DIR / "preds_all_valid.parquet")
pt = pd.read_parquet(OUT_DIR / "preds_all_test.parquet")
MODELS = [c for c in pt.columns if c not in ("case_id", "WEEK_NUM", "target")]
print("Mo hinh co du doan:", MODELS)

## 1. Khoảng tin cậy bootstrap và kiểm định DeLong (tập test)

In [ ]:
rows = []
for m in MODELS:
    auc, lo, hi = utils.bootstrap_ci(pt["target"], pt[m], n=500, seed=SEED)
    rows.append({"mo_hinh": m, "AUC": auc, "AUC_CI95_thap": lo, "AUC_CI95_cao": hi,
                 "Gini": 2 * auc - 1, "Gini_CI95_thap": 2 * lo - 1, "Gini_CI95_cao": 2 * hi - 1})
ci = pd.DataFrame(rows).set_index("mo_hinh").sort_values("AUC", ascending=False)
ci.to_csv(OUT_DIR / "auc_bootstrap_ci.csv"); display(ci.round(4))

In [ ]:
best = ci.index[0]
dl = []
for m in MODELS:
    if m == best: continue
    r = utils.delong_test(pt["target"], pt[best], pt[m]); r["so_sanh"] = f"{best}  vs  {m}"; dl.append(r)
dl = pd.DataFrame(dl).set_index("so_sanh").sort_values("chenh")
dl["co_y_nghia_5%"] = dl["p_value"] < 0.05
dl.to_csv(OUT_DIR / "delong_tests.csv"); display(dl.round(5))
summary["Mo hinh tot nhat (test)"] = best
summary["AUC test [CI95]"] = "%.4f [%.4f; %.4f]" % tuple(ci.loc[best, ["AUC", "AUC_CI95_thap", "AUC_CI95_cao"]])

## 2. Chỉ số ổn định Gini theo tuần (định nghĩa của cuộc thi)
`stability = mean(Gini tuần) + 88 × min(0, độ dốc) − 0,5 × std(phần dư)` – phạt mô hình có Gini giảm dần theo thời gian hoặc dao động mạnh giữa các tuần.

In [ ]:
st = pd.DataFrame({m: utils.gini_stability(pt.assign(score=pt[m])) for m in MODELS}).T.sort_values("stability", ascending=False)
st.to_csv(OUT_DIR / "gini_stability.csv"); display(st.round(4))
summary["Gini stability (test)"] = round(st.loc[best, "stability"], 4)

gw = utils.gini_by_week(pt.assign(score=pt[best]))
ax = gw.plot(marker="o", figsize=(8, 3), title=f"Gini theo tuan (test) - {best}")
x = np.arange(len(gw)); a, b = np.polyfit(x, gw.values, 1); ax.plot(gw.index, a * x + b, "--", c="gray"); plt.show()

## 3. Kiểm định cửa sổ thời gian (rolling-origin)
Bốn khối kiểm định liên tiếp, mỗi khối huấn luyện chỉ trên dữ liệu **trước** khối đó, chừa **khoảng đệm 2 tuần**:
- **Cửa sổ mở rộng**: train từ tuần 0 → trước khoảng đệm.
- **Cửa sổ trượt**: chỉ `SLIDE` tuần gần nhất.

Early stopping dùng 8 tuần cuối của chính cửa sổ huấn luyện (không chạm khối kiểm định). Để giới hạn thời gian chạy, notebook dùng `learning_rate = 0.08` nên AUC có thể thấp hơn mô hình chính đôi chút; mục đích là **so sánh giữa các khối**, không phải thay mô hình chính.

In [ ]:
import joblib
HAS_FINAL = (OUT_DIR / "final_model.joblib").exists()
if HAS_FINAL:                                    # mo hinh cuoi: dac trung goc + credit bureau (notebook 05)
    fpack = joblib.load(OUT_DIR / "final_model.joblib"); feats = fpack["features"]
    df = pl.read_parquet(OUT_DIR / "features_v2.parquet").to_pandas()
else:
    df = pl.read_parquet(OUT_DIR / "features_v1.parquet").to_pandas(); feats = [c for c in df.columns if c not in META]
print("Bo dac trung kiem dinh:", "features_v2 (mo hinh cuoi)" if HAS_FINAL else "features_v1", "|", len(feats), "bien")
BLOCKS = [(40, 52), (52, 64), (64, 78), (78, 92)]
GAP, SLIDE, ES_WEEKS = 2, 40, 8
P_FAST = dict(n_estimators=2000, learning_rate=0.08, num_leaves=31, min_child_samples=200, subsample=0.8, subsample_freq=1,
              colsample_bytree=0.6, reg_lambda=5.0, random_state=SEED, n_jobs=-1, verbose=-1, metric="auc")

def fit_eval(w_start, w_end, block):
    tr = df[(df.WEEK_NUM >= w_start) & (df.WEEK_NUM < w_end - ES_WEEKS)]
    es = df[(df.WEEK_NUM >= w_end - ES_WEEKS) & (df.WEEK_NUM < w_end)]
    te = df[(df.WEEK_NUM >= block[0]) & (df.WEEK_NUM < block[1])]
    m = lgb.LGBMClassifier(**P_FAST).fit(tr[feats], tr["target"], eval_set=[(es[feats], es["target"])],
                                         callbacks=[lgb.early_stopping(100, verbose=False)])
    p = m.predict_proba(te[feats])[:, 1]
    return {"tuan_train": f"{w_start}-{w_end - 1}", "so_ho_so_train": len(tr) + len(es), "tuan_kiem_dinh": f"{block[0]}-{block[1] - 1}",
            "AUC": roc_auc_score(te["target"], p), "Gini_tuan_tb": utils.gini_by_week(te.assign(score=p)).mean(),
            "ty_le_vo_no": te["target"].mean(), "so_cay": m.best_iteration_}

roll = []
for blk in BLOCKS:
    end = blk[0] - GAP
    roll.append({"cua_so": "mo rong", **fit_eval(0, end, blk)})
    roll.append({"cua_so": f"truot {SLIDE} tuan", **fit_eval(max(0, end - SLIDE), end, blk)})
    print("Xong khoi", blk)
roll = pd.DataFrame(roll); roll.to_csv(OUT_DIR / "rolling_origin.csv", index=False)
display(roll.round(4))
agg = roll.groupby("cua_so")["AUC"].agg(["mean", "std", "min"]); display(agg.round(4))
summary["Rolling AUC (mo rong) tb +- std"] = "%.4f +- %.4f" % tuple(agg.loc["mo rong", ["mean", "std"]])

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for w, g in roll.groupby("cua_so"): ax.plot(g["tuan_kiem_dinh"], g["AUC"], "o-", label=w)
ax.set_ylabel("AUC"); ax.set_xlabel("Khoi tuan kiem dinh"); ax.set_title("Kiem dinh cua so thoi gian"); ax.legend(); plt.show()

## 4. Hiệu chỉnh xác suất (mô hình chọn ở notebook 03)
Mô hình dùng để ra quyết định hạn mức cần **xác suất đúng mức**, không chỉ xếp hạng tốt. Isotonic được fit trên valid, đánh giá trên test.
Hiệu chỉnh tốt: ECE gần 0, hệ số độ dốc gần 1, hệ số chặn gần 0.

In [ ]:
tag = "final" if HAS_FINAL else ""
pr_v = pd.read_parquet(OUT_DIR / f"preds{'_' + tag if tag else ''}_valid.parquet")
pr_t = pd.read_parquet(OUT_DIR / f"preds{'_' + tag if tag else ''}_test.parquet")
iso = IsotonicRegression(y_min=1e-4, y_max=1 - 1e-4, out_of_bounds="clip").fit(pr_v["pd_raw"], pr_v["target"])
cal = pd.DataFrame({"truoc hieu chinh": utils.calibration_stats(pr_t["target"], pr_t["pd_raw"]),
                    "sau isotonic": utils.calibration_stats(pr_t["target"], iso.predict(pr_t["pd_raw"]))}).T
cal.to_csv(OUT_DIR / "calibration_stats.csv"); display(cal.round(4))
summary["ECE sau hieu chinh"] = round(cal.loc["sau isotonic", "ECE"], 5)

# Hang rui ro voi mo hinh cuoi (diem cat xac dinh tren valid, giong notebook 04)
pr_v["pd"], pr_t["pd"] = iso.predict(pr_v["pd_raw"]), iso.predict(pr_t["pd_raw"])
FACTOR = 20 / np.log(2); OFFSET = 600 - FACTOR * np.log(50)
for d in (pr_v, pr_t): d["score"] = OFFSET + FACTOR * np.log((1 - d["pd"]) / d["pd"])
edges = np.concatenate([[-np.inf], np.unique(np.quantile(pr_v["score"], [0.10, 0.30, 0.60, 0.85])), [np.inf]])
labels = ["E", "D", "C", "B", "A"][-(len(edges) - 1):]
pr_t["band"] = pd.cut(pr_t["score"], edges, labels=labels)
bt = pr_t.groupby("band", observed=True).agg(so_ho_so=("target", "size"), ty_le_vo_no=("target", "mean"), pd_tb=("pd", "mean"))
bt["ty_trong"] = bt["so_ho_so"] / len(pr_t); bt.to_csv(OUT_DIR / "final_bands.csv")
print("Ty le vo no giam don dieu E -> A:", bool(bt["ty_le_vo_no"].is_monotonic_decreasing)); display(bt.round(4))

## 5. Trôi dữ liệu: adversarial validation và PSI từng biến
Huấn luyện một mô hình phân biệt hồ sơ **train (tuần 0–63)** với hồ sơ **test (tuần 78–91)**. AUC ≈ 0,5: hai giai đoạn giống nhau; AUC cao: có biến thay đổi phân phối theo thời gian – cần theo dõi khi triển khai.

In [ ]:
tr_w = df[df.WEEK_NUM <= 63].sample(150_000, random_state=SEED) if (df.WEEK_NUM <= 63).sum() > 150_000 else df[df.WEEK_NUM <= 63]
te_w = df[df.WEEK_NUM >= 78].sample(min(150_000, (df.WEEK_NUM >= 78).sum()), random_state=SEED)
adv = pd.concat([tr_w.assign(is_test=0), te_w.assign(is_test=1)]).sample(frac=1, random_state=SEED)
cut = int(len(adv) * 0.7)
am = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, colsample_bytree=0.6, verbose=-1, random_state=SEED)
am.fit(adv[feats].iloc[:cut], adv["is_test"].iloc[:cut])
adv_auc = roc_auc_score(adv["is_test"].iloc[cut:], am.predict_proba(adv[feats].iloc[cut:])[:, 1])
print("AUC adversarial (train vs test): %.4f" % adv_auc)
summary["AUC adversarial"] = round(adv_auc, 4)

num = [c for c in feats if pd.api.types.is_numeric_dtype(df[c])]
psi = pd.Series({c: utils.psi(tr_w[c].dropna(), te_w[c].dropna()) for c in num if tr_w[c].notna().sum() > 100 and te_w[c].notna().sum() > 100})
imp_adv = pd.Series(am.booster_.feature_importance("gain"), index=feats)
drift = pd.DataFrame({"PSI": psi, "gain_adversarial_%": 100 * imp_adv / imp_adv.sum()}).sort_values("gain_adversarial_%", ascending=False)
drift.to_csv(OUT_DIR / "feature_drift.csv")
print("So bien PSI > 0.25 (troi manh):", int((psi > 0.25).sum()), "| 0.1-0.25 (can theo doi):", int(((psi > 0.1) & (psi <= 0.25)).sum()))
display(drift.head(15).round(4))

## 6. Kiểm tra rò rỉ của quy trình: hoán vị nhãn
Nếu quy trình không rò rỉ thông tin nhãn, mô hình huấn luyện trên **nhãn bị xáo trộn ngẫu nhiên** phải cho AUC test ≈ 0,5.

In [ ]:
tr_p = df[df.WEEK_NUM <= 63].sample(min(300_000, (df.WEEK_NUM <= 63).sum()), random_state=SEED)
y_perm = np.random.default_rng(SEED).permutation(tr_p["target"].values)
te_p = df[df.WEEK_NUM >= 78]
pm = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, colsample_bytree=0.6, verbose=-1, random_state=SEED)
pm.fit(tr_p[feats], y_perm)
perm_auc = roc_auc_score(te_p["target"], pm.predict_proba(te_p[feats])[:, 1])
print("AUC test voi nhan hoan vi: %.4f (ky vong ~0.5)" % perm_auc)
summary["AUC nhan hoan vi"] = round(perm_auc, 4)

## 6b. Phân tích độ nhạy: bỏ thuộc tính nhạy cảm (giới tính)
Mô hình chính có dùng giới tính khách hàng. Trong thực tế, nhiều quy định cho vay hạn chế dùng thuộc tính nhạy cảm để ra quyết định.
Ô dưới huấn luyện lại mô hình cuối **sau khi bỏ** các cột có mô tả về giới tính (không thêm biến nào), so sánh AUC bằng DeLong và so sánh tỷ lệ duyệt theo giới tính ở cùng ngưỡng.

In [ ]:
import re
defs = utils.load_definitions()
sens = [c for c in feats if re.search(r"\bgender\b|\bsex\b", defs.get(utils.base_name(c), ""), re.I)]
print("Cot nhay cam bi bo:", sens)
if sens:
    tr_ = df[df.WEEK_NUM <= 63]; va_ = df[(df.WEEK_NUM >= 64) & (df.WEEK_NUM <= 77)]; te_ = df[df.WEEK_NUM >= 78]
    fs = [c for c in feats if c not in sens]
    P_MAIN = dict(P_FAST, learning_rate=0.03, n_estimators=3000)
    m_s = lgb.LGBMClassifier(**P_MAIN).fit(tr_[fs], tr_["target"], eval_set=[(va_[fs], va_["target"])],
                                           callbacks=[lgb.early_stopping(200, verbose=False)])
    p_s = m_s.predict_proba(te_[fs])[:, 1]
    p_f = (fpack["model"].predict_proba(te_[feats])[:, 1] if HAS_FINAL else pt.set_index("case_id").loc[te_["case_id"], best].values)
    r = utils.delong_test(te_["target"], p_f, p_s)
    print("AUC co gioi tinh %.4f | khong gioi tinh %.4f | chenh %.4f | p = %.3g" % (r["AUC_1"], r["AUC_2"], r["chenh"], r["p_value"]))
    summary["AUC bo gioi tinh (chenh, p)"] = "%.4f (%.4f, p=%.2g)" % (r["AUC_2"], r["chenh"], r["p_value"])
    g = te_[sens[0]].astype(str)
    for name, p in [("co gioi tinh", p_f), ("khong gioi tinh", p_s)]:
        thr = np.quantile(p, 0.90)                               # duyet 90% ho so rui ro thap nhat
        ok = pd.Series(p <= thr, index=te_.index)
        t = pd.DataFrame({"ty_le_duyet": ok.groupby(g).mean(), "vo_no_nhom_duyet": te_["target"][ok].groupby(g[ok]).mean()})
        print("\n", name); display(t.round(4))

## 7. Tổng hợp kiểm định

In [ ]:
s = pd.Series(summary, name="gia_tri").to_frame(); s.to_csv(OUT_DIR / "validation_summary.csv"); s

## 8. Nhận xét (tự viết)
- Độ chính xác: AUC/Gini và khoảng tin cậy; chênh lệch với mô hình nền có ý nghĩa thống kê không (DeLong): ...
- Độ ổn định: Gini stability, kết quả cửa sổ mở rộng vs cửa sổ trượt (có nên huấn luyện lại định kỳ không): ...
- Hiệu chỉnh: PD sau isotonic có dùng được cho tính tổn thất kỳ vọng / hạn mức không: ...
- Trôi dữ liệu và rò rỉ: AUC adversarial, các biến trôi mạnh cần giám sát; kết quả hoán vị nhãn: ...
- Kết luận về khả năng ứng dụng: ...

---
# PHẦN 7: 08_Beyond_Obvious_Risk

# Notebook 08 – Điều gì đi cùng rủi ro cao, **ngoài** các dấu hiệu túng thiếu hiển nhiên?

Trễ hạn, vay nhiều nơi, nợ nặng, đảo nợ là những dấu hiệu túng thiếu mà ai cũng biết. Câu hỏi ở đây: **sau khi đã tính đến các dấu hiệu đó**, còn đặc điểm nào khác của hồ sơ đi cùng với vỡ nợ?

Phương pháp (chỉ dùng cột có sẵn, không tạo biến mới):
1. **Mô hình "hiển nhiên"**: LightGBM chỉ dùng 4 nhóm hành vi hiển nhiên → xác suất vỡ nợ *kỳ vọng* của mỗi hồ sơ.
2. **Giá trị gia tăng**: so sánh với mô hình cuối (thêm mọi đặc trưng còn lại) bằng kiểm định DeLong.
3. **SHAP**: xếp hạng các đặc trưng *không hiển nhiên* theo đóng góp vào dự báo của mô hình cuối, kèm chiều tác động.
4. **Rủi ro vượt kỳ vọng**: với từng đặc trưng không hiển nhiên, so sánh số vỡ nợ **quan sát** với số vỡ nợ **kỳ vọng** từ mô hình hiển nhiên theo từng nhóm giá trị; kiểm định z ( |z| > 3 ⇒ chênh lệch có ý nghĩa thống kê ).
5. **Rủi ro ẩn ở khách "trông an toàn"**: trong nhóm 60% hồ sơ có rủi ro hiển nhiên thấp nhất, cây quyết định nông (độ sâu 3) tìm các phân khúc có tỷ lệ vỡ nợ cao bất thường – ra quy tắc dễ đọc.

**Lưu ý diễn giải**: đây là **tương quan**, không phải nhân quả. Thuộc tính nhạy cảm (giới tính, tuổi…) được đánh dấu riêng: chúng có thể tương quan với rủi ro nhưng không nên dùng để từ chối khách hàng.

In [ ]:
# Giai phong bo nho con lai tu cac phan truoc (chay TRUOC khi doc du lieu cua phan nay)
import utils
utils.free_memory(globals())

In [ ]:
import warnings, re; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, polars as pl, joblib
import matplotlib.pyplot as plt
import utils
from utils import META
import lightgbm as lgb
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score
from sklearn.tree import DecisionTreeClassifier, export_text

OUT_DIR, SEED = utils.get_out_dir(), 42
pd.set_option("display.max_colwidth", 80)
defs = utils.load_definitions()
pack = joblib.load(OUT_DIR / "final_model.joblib"); final, feats = pack["model"], pack["features"]
df = pl.read_parquet(OUT_DIR / "features_v2.parquet").to_pandas()
grp = pd.read_csv(OUT_DIR / "behavior_groups.csv", index_col=0)["nhom"]
OBVIOUS = {"tre_han", "vay_nhieu_noi", "ganh_nang_no", "dao_no"}
obv = [c for c in feats if grp.get(c) in OBVIOUS]
other = [c for c in feats if c not in obv]
desc = utils.describe(feats, defs)
SENS = re.compile(r"\bgender\b|\bsex\b|birth|\bage\b|marital|religio|nationality", re.I)
sensitive = [c for c in other if SENS.search(desc[c] or "")]
train, valid, test = utils.time_split(df)
print("Dac trung hien nhien:", len(obv), "| khong hien nhien:", len(other), "| nhay cam:", sensitive)

## 1–2. Mô hình chỉ dùng dấu hiệu hiển nhiên và giá trị gia tăng của phần còn lại

In [ ]:
P = dict(n_estimators=3000, learning_rate=0.03, num_leaves=31, min_child_samples=200, subsample=0.8, subsample_freq=1,
         colsample_bytree=0.6, reg_lambda=5.0, random_state=SEED, n_jobs=-1, verbose=-1, metric="auc")
m_obv = lgb.LGBMClassifier(**P).fit(train[obv], train["target"], eval_set=[(valid[obv], valid["target"])],
                                    callbacks=[lgb.early_stopping(200, verbose=False)])
iso = IsotonicRegression(y_min=1e-4, y_max=1 - 1e-4, out_of_bounds="clip").fit(m_obv.predict_proba(valid[obv])[:, 1], valid["target"])
for d in (train, valid, test):
    d["p_obv"] = iso.predict(m_obv.predict_proba(d[obv])[:, 1])      # xac suat vo no ky vong tu dau hieu hien nhien
p_final = final.predict_proba(test[feats])[:, 1]
dl = utils.delong_test(test["target"], p_final, test["p_obv"])
print("AUC test - chi dau hieu hien nhien: %.4f | mo hinh cuoi: %.4f | chenh %.4f (z = %.1f, p = %.2g)" % (
      dl["AUC_2"], dl["AUC_1"], dl["chenh"], dl["z"], dl["p_value"]))
pd.Series({"AUC_hien_nhien": dl["AUC_2"], "AUC_cuoi": dl["AUC_1"], "chenh": dl["chenh"], "p_value": dl["p_value"]}).to_csv(OUT_DIR / "obvious_vs_final.csv")

## 3. Đặc trưng không hiển nhiên đóng góp nhiều nhất (SHAP, mô hình cuối)
`chieu`: **+** nghĩa là giá trị càng cao thì dự báo rủi ro càng cao (tương quan Spearman giữa giá trị và SHAP); với biến phân loại xem bảng ở mục 4.

In [ ]:
import shap
Xs = test.sample(min(len(test), 20_000), random_state=SEED)
sv = shap.TreeExplainer(final.booster_).shap_values(Xs[feats]); sv = sv[1] if isinstance(sv, list) else sv
S = pd.DataFrame(sv, columns=feats, index=Xs.index)
imp = S.abs().mean()
rows = []
for c in other:
    x = Xs[c]
    sign = (pd.Series(pd.to_numeric(x, errors="coerce")).corr(S[c], method="spearman")
            if pd.api.types.is_numeric_dtype(x) else np.nan)
    rows.append({"dac_trung": c, "ty_trong_SHAP_%": 100 * imp[c] / imp.sum(),
                 "chieu": "" if pd.isna(sign) else ("+" if sign > 0 else "-"),
                 "nhay_cam": c in sensitive, "mo_ta": desc[c]})
top_other = pd.DataFrame(rows).sort_values("ty_trong_SHAP_%", ascending=False).set_index("dac_trung")
print("Ty trong SHAP: hien nhien %.1f%% | khong hien nhien %.1f%%" % (100 * imp[obv].sum() / imp.sum(), 100 * imp[other].sum() / imp.sum()))
top_other.to_csv(OUT_DIR / "non_obvious_shap.csv"); top_other.head(20).round(2)

## 4. Rủi ro vượt kỳ vọng: quan sát so với kỳ vọng từ dấu hiệu hiển nhiên (tập test)
Với mỗi nhóm giá trị của một đặc trưng: `O` = số vỡ nợ thực tế, `E` = tổng xác suất kỳ vọng từ mô hình hiển nhiên,
`O/E` > 1 ⇒ nhóm này vỡ nợ **nhiều hơn** mức mà các dấu hiệu túng thiếu giải thích được. `z = (O − E) / √Σp(1−p)`; |z| > 3 xem là có ý nghĩa thống kê (đã tính đến việc kiểm định nhiều nhóm).

In [ ]:
def excess_table(d, col, q=5, min_n=500):
    x = d[col]
    if pd.api.types.is_numeric_dtype(x):
        b = pd.qcut(x, q, duplicates="drop")
        b = b.cat.add_categories("thieu du lieu").fillna("thieu du lieu")       # giu thu tu khoang gia tri
    else:
        top = x.astype(str).value_counts().index[:8]
        b = x.astype(str).where(x.astype(str).isin(top), "khac").where(x.notna(), "thieu du lieu")
    g = d.assign(b=b).groupby("b", observed=True, sort=True).agg(n=("target", "size"), O=("target", "sum"), E=("p_obv", "sum"),
                                      V=("p_obv", lambda p: (p * (1 - p)).sum()))
    g = g[g["n"] >= min_n]
    g["ty_le_vo_no"] = g["O"] / g["n"]; g["O/E"] = g["O"] / g["E"]; g["z"] = (g["O"] - g["E"]) / np.sqrt(g["V"])
    return g.drop(columns="V")

TOP_N = 12
out, summ = {}, []
for c in top_other.index[:TOP_N]:
    t = excess_table(test, c); out[c] = t
    k = t["z"].abs().idxmax()
    summ.append({"dac_trung": c, "nhom_chenh_nhieu_nhat": k, "O/E": t.loc[k, "O/E"], "z": t.loc[k, "z"],
                 "ty_le_vo_no_nhom": t.loc[k, "ty_le_vo_no"], "nhay_cam": c in sensitive, "mo_ta": desc[c]})
summ = pd.DataFrame(summ).set_index("dac_trung")
summ["co_y_nghia"] = summ["z"].abs() > 3
summ.to_csv(OUT_DIR / "non_obvious_excess_risk.csv")
print("Ty le vo no chung (test): %.2f%%" % (100 * test["target"].mean())); display(summ.round(3))

In [ ]:
# Bang chi tiet cho 4 dac trung co chenh lech lon nhat
for c in summ["z"].abs().sort_values(ascending=False).index[:4]:
    print(f"\n=== {c}: {desc[c]}" + ("   [THUOC TINH NHAY CAM]" if c in sensitive else ""))
    display(out[c][["n", "ty_le_vo_no", "O", "E", "O/E", "z"]].round(3))

## 5. Rủi ro ẩn ở khách hàng "trông an toàn"
Nhóm "trông an toàn" = 60% hồ sơ có xác suất kỳ vọng từ dấu hiệu hiển nhiên thấp nhất (ngưỡng lấy từ train).
Cây quyết định độ sâu 3, chỉ dùng đặc trưng **số** không hiển nhiên và **không nhạy cảm**, học trên train (nhóm an toàn) và đánh giá trên test (nhóm an toàn).
Giá trị thiếu được thay bằng −999 để cây tách riêng nhóm thiếu dữ liệu (không suy diễn giá trị).

In [ ]:
thr = np.quantile(train["p_obv"], 0.60)
safe_tr, safe_te = train[train["p_obv"] <= thr], test[test["p_obv"] <= thr]
num_other = [c for c in top_other.index if c not in sensitive and pd.api.types.is_numeric_dtype(df[c])][:30]
tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=max(500, len(safe_tr) // 200), class_weight=None, random_state=SEED)
tree.fit(safe_tr[num_other].fillna(-999), safe_tr["target"])
leaf_te = tree.apply(safe_te[num_other].fillna(-999))
base_safe = safe_te["target"].mean()
leaves = (safe_te.assign(la=leaf_te).groupby("la").agg(so_ho_so=("target", "size"), ty_le_vo_no=("target", "mean"), pd_ky_vong=("p_obv", "mean")))
leaves["lift_trong_nhom_an_toan"] = leaves["ty_le_vo_no"] / base_safe
leaves = leaves.sort_values("ty_le_vo_no", ascending=False)
print("Nhom trong an toan (test): %d ho so | ty le vo no %.2f%% (chung %.2f%%) | AUC cay tren test: %.3f" % (
      len(safe_te), 100 * base_safe, 100 * test["target"].mean(), roc_auc_score(safe_te["target"], tree.predict_proba(safe_te[num_other].fillna(-999))[:, 1])))
leaves.to_csv(OUT_DIR / "hidden_risk_leaves.csv"); display(leaves.round(4))
print(export_text(tree, feature_names=num_other, show_weights=False, decimals=1))
print("Mo ta cac bien trong cay:"); display(desc[[c for c in num_other if c in export_text(tree, feature_names=num_other)]].rename("mo_ta").to_frame())

## 6. Nhận xét (tự viết)
- Phần rủi ro mà dấu hiệu túng thiếu hiển nhiên **không** giải thích được lớn đến đâu (chênh AUC, tỷ trọng SHAP): ...
- Các đặc điểm không hiển nhiên nổi bật (sản phẩm/kênh, thu nhập – nghề nghiệp, thông tin liên hệ dùng chung, thời gian quan hệ…) và nhóm giá trị có O/E cao, |z| > 3: ...
- Phân khúc rủi ro ẩn trong nhóm "trông an toàn" và cách dùng trong thực tế (xác minh thêm, hạn mức thấp hơn, theo dõi sau giải ngân): ...
- Thuộc tính nhạy cảm: có tương quan không; vì sao không nên dùng để từ chối: ...
- Hạn chế: tương quan không phải nhân quả; tên một số giá trị phân loại đã bị ban tổ chức mã hoá.

---
# PHẦN 8: 09_Extra_Tables_Experiment

# Notebook 09 – Thử nghiệm mở rộng: bảng thuế và lịch sử hồ sơ vay độ sâu 2

Các nghiên cứu trên cùng bộ dữ liệu có dùng thêm **dữ liệu cơ quan thuế** (`tax_registry_a/b/c`) và **bảng độ sâu 2** (nhiều bản ghi trên mỗi hồ sơ cũ, đánh chỉ số bởi `num_group1`, `num_group2`). Notebook này kiểm tra, **có đối chứng**, việc thêm các bảng đó có cải thiện mô hình cuối không.

- Chỉ dùng cột thật của bảng, gộp về 1 dòng/hồ sơ bằng cùng hàm như notebook 02 (số bản ghi, max, mean, giá trị đầu); ngày đổi thành số ngày so với ngày quyết định.
- Áp dụng cùng các quy tắc chống rò rỉ: loại cột ngày phát sinh sau quyết định và cột trôi độ sẵn có (train vs valid, không dùng test).
- So sánh với mô hình cuối (notebook 05) trên cùng tập test bằng **kiểm định DeLong** và Gini theo tuần. Chỉ giữ thay đổi nếu cải thiện có ý nghĩa thống kê.
- `credit_bureau_a_2` không dùng vì rất lớn (hàng trăm triệu dòng), dễ vượt bộ nhớ khi nộp bài.

Notebook này **không** thay đổi file nộp bài (notebook 07 vẫn dùng mô hình cuối của notebook 05).

In [ ]:
# Giai phong bo nho con lai tu cac phan truoc (chay TRUOC khi doc du lieu cua phan nay)
import utils
utils.free_memory(globals())

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, polars as pl, joblib
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from utils import META
OUT_DIR, SEED = utils.get_out_dir(), 42
defs = utils.load_definitions()
EXTRA = ["tax_registry_a_1", "tax_registry_b_1", "tax_registry_c_1", "applprev_2"]

base = utils.load_base()
blocks = []
for t in EXTRA:
    try:
        utils.table_files(t)
    except FileNotFoundError as e:
        print("Bo qua:", e); continue
    f = utils.collect(utils.depth1_features(t, base))
    print(t, "->", f.shape)
    blocks.append(f)
ext = blocks[0]
for f in blocks[1:]:
    ext = ext.join(f, on="case_id", how="full", coalesce=True)
ext = utils.prune_features(ext, max_null=0.95)
print("Dac trung moi:", ext.width - 1)

In [ ]:
# Chong ro ri: cot ngay sau quyet dinh + troi do san co (train vs valid)
df2 = pl.read_parquet(OUT_DIR / "features_v2.parquet").select(["case_id", "WEEK_NUM"])
new = [c for c in ext.columns if c != "case_id"]
tmp = df2.join(ext, on="case_id", how="left")
fut = utils.future_date_columns(tmp, new, defs, min_share=0.01)
drop = fut.loc[fut["loai"], "cot"].tolist()
weeks = np.sort(tmp["WEEK_NUM"].unique()); n_w = len(weeks)
w_tr, w_va = weeks[: int(n_w * 0.70)].tolist(), weeks[int(n_w * 0.70): int(n_w * 0.85)].tolist()
n_tr = tmp.filter(pl.col("WEEK_NUM").is_in(w_tr)).select([pl.col(c).is_null().mean().alias(c) for c in new]).row(0)
n_va = tmp.filter(pl.col("WEEK_NUM").is_in(w_va)).select([pl.col(c).is_null().mean().alias(c) for c in new]).row(0)
drop += [c for c, a, b in zip(new, n_tr, n_va) if abs(b - a) >= 0.30]
ext = ext.drop(sorted(set(drop)))
new = [c for c in ext.columns if c != "case_id"]
print("Loai:", len(set(drop)), "| giu lai:", len(new))
del tmp, df2

In [ ]:
pack = joblib.load(OUT_DIR / "final_model.joblib"); feats = pack["features"]
df = pl.read_parquet(OUT_DIR / "features_v2.parquet").join(ext, on="case_id", how="left").to_pandas()
train, valid, test = utils.time_split(df)
P = dict(n_estimators=3000, learning_rate=0.03, num_leaves=31, min_child_samples=200, subsample=0.8, subsample_freq=1,
         colsample_bytree=0.6, reg_lambda=5.0, random_state=SEED, n_jobs=-1, verbose=-1, metric="auc")
m = lgb.LGBMClassifier(**P).fit(train[feats + new], train["target"], eval_set=[(valid[feats + new], valid["target"])],
                                callbacks=[lgb.early_stopping(200, verbose=False)])
p_new = m.predict_proba(test[feats + new])[:, 1]
p_old = pack["model"].predict_proba(test[feats])[:, 1]
dl = utils.delong_test(test["target"], p_new, p_old)
res = pd.DataFrame({
    "mo_hinh": ["Mo hinh cuoi (notebook 05)", "+ bang thue + applprev_2"],
    "so_bien": [len(feats), len(feats) + len(new)],
    "AUC_test": [dl["AUC_2"], dl["AUC_1"]],
    "Gini_tuan_tb": [utils.gini_by_week(test.assign(score=p_old)).mean(), utils.gini_by_week(test.assign(score=p_new)).mean()],
    "stability": [utils.gini_stability(test.assign(score=p_old))["stability"], utils.gini_stability(test.assign(score=p_new))["stability"]],
}).set_index("mo_hinh")
res.to_csv(OUT_DIR / "extra_tables_experiment.csv"); display(res.round(4))
print("DeLong: chenh AUC = %.4f | z = %.2f | p = %.3g -> %s" % (dl["chenh"], dl["z"], dl["p_value"],
      "cai thien co y nghia thong ke" if dl["p_value"] < 0.05 and dl["chenh"] > 0 else "khong cai thien co y nghia"))
imp = pd.Series(m.booster_.feature_importance("gain"), index=feats + new)
print("Ty trong gain cua bien moi: %.1f%%" % (100 * imp[new].sum() / imp.sum()))
display(pd.DataFrame({"gain_%": 100 * imp[new] / imp.sum(), "mo_ta": utils.describe(new, defs)}).sort_values("gain_%", ascending=False).head(10).round(2))

## Nhận xét (tự viết)
- Thêm bảng thuế và lịch sử hồ sơ vay độ sâu 2 có cải thiện AUC / độ ổn định có ý nghĩa thống kê không: ...
- Nếu có: cân nhắc chi phí (thời gian, bộ nhớ, độ phức tạp vận hành) so với mức cải thiện; nếu không: ghi nhận là kết quả âm tính có giá trị.

---
# PHẦN 9: 07_Submission

# Notebook 07 – Dự đoán cho tập test của cuộc thi (nộp bài Kaggle)

Đây là **kiểm định độc lập bên ngoài**: khi nộp bài, Kaggle chạy lại toàn bộ notebook trên **tập test ẩn có nhãn thật** (người nộp không nhìn thấy nhãn) và chấm bằng chỉ số ổn định Gini theo tuần.

Quy trình áp dụng **đúng** các bước như với dữ liệu train: cùng hàm gộp bảng, cùng 3 tỷ lệ, cùng hàm gộp `credit_bureau_a_1`, cùng danh sách đặc trưng và **mô hình cuối** (đặc trưng gốc + credit bureau, notebook 05). Không có đặc trưng hay nhãn nào được tạo thêm.
Kết quả: `submission.csv` gồm `case_id`, `score` (xác suất vỡ nợ).

In [ ]:
# Giai phong bo nho con lai tu cac phan truoc (chay TRUOC khi doc du lieu cua phan nay)
import utils
utils.free_memory(globals())

In [ ]:
import numpy as np, pandas as pd, polars as pl, joblib
import utils

OUT_DIR = utils.get_out_dir()
USE_FINAL = (OUT_DIR / "final_model.joblib").exists()
pack = joblib.load(OUT_DIR / ("final_model.joblib" if USE_FINAL else "best_model.joblib"))
model, feats = pack["model"], pack["features"]
print("Mo hinh:", pack["name"], "| so dac trung:", len(feats))

utils.SPLIT = "test"                                   # doc test_*.parquet
try:
    raw = utils.build_features(["static_0", "static_cb_0"], ["applprev_1", "person_1"], None)
    if USE_FINAL:
        cbf, _ = utils.credit_bureau_features(utils.load_base(), utils.load_definitions())
        raw = raw.join(cbf, on="case_id", how="left")
finally:
    utils.SPLIT = "train"
print("Tap test:", raw.shape)

def add_ratio(df, num, den, name):                     # giong het notebook 02
    if num in df.columns and den in df.columns:
        return df.with_columns((pl.col(num) / pl.when(pl.col(den) > 0).then(pl.col(den))).alias(name))
    return df
raw = add_ratio(raw, "annuity_780A", "maininc_215A", "ratio_annuity_income")
raw = add_ratio(raw, "credamount_770A", "maininc_215A", "ratio_credit_income")
raw = add_ratio(raw, "annuity_780A", "credamount_770A", "ratio_annuity_credit")

In [ ]:
# Dua ve dung cot va kieu du lieu nhu luc huan luyen
schema = pl.read_parquet_schema(OUT_DIR / ("features_v2.parquet" if USE_FINAL else "features_v1.parquet"))
print("Cot thieu trong tap test (dien rong):", len([c for c in feats if c not in raw.columns]))
parts, CHUNK = [], 100_000                              # du doan theo khoi de tiet kiem bo nho
for s0 in range(0, raw.height, CHUNK):
    X = utils.align_features(raw.slice(s0, CHUNK).to_pandas(), feats, schema)
    parts.append(pd.DataFrame({"case_id": X["case_id"].astype("int64"), "score": model.predict_proba(X[feats])[:, 1]}))
    del X
sub = pd.concat(parts, ignore_index=True)
sub.to_csv(OUT_DIR / "submission.csv", index=False)
print("Da ghi", OUT_DIR / "submission.csv", sub.shape)
sub.describe()